# Base Models

Standalone ML, DL, and Transformer experiments, linguistic baselines, and user-level evaluation. Data preparation and linguistic analysis are in notebook 00.


## 1. Configuration

Experiment paths, seeds, and execution options.


In [1]:
from pathlib import Path
import gc
import hashlib
import json
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import sklearn
import joblib
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
from IPython.display import display

BASE = Path('/content/drive_research/MyDrive/thesis/saved_results')
MULTISEED = BASE / '17_conference_revision/multiseed'
STACKING = BASE / '17_conference_revision/stacking_oof'
SEEDS = [42, 43, 44, 45, 46]
ENABLE_TRAINING = False
ENABLE_META_FIT = False
ENABLE_THRESHOLD_SEARCH = False
TRAINING_STAGE = None

METRICS = ['accuracy', 'precision_ASD', 'recall_ASD', 'f1_ASD',
           'macro_f1', 'recall_non_ASD', 'f1_non_ASD', 'roc_auc']

def require_files(paths):
    missing = [str(Path(p)) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError('Missing files:\n' + '\n'.join(missing))

def read_json(path):
    require_files([path])
    return json.loads(Path(path).read_text())

def require_training(stage, meta=False):
    if ENABLE_TRAINING is not True or TRAINING_STAGE != stage:
        raise RuntimeError('Training is disabled.')
    if meta and not (ENABLE_META_FIT is True and ENABLE_THRESHOLD_SEARCH is True):
        raise RuntimeError('Meta fitting and threshold search are disabled.')
    if meta:
        seeds = [42] if stage == 'meta_42' else [43,44,45,46]
        targets = [STACKING/f'seed_{s}'/f for s in seeds
                   for f in ['PRIVATE_meta_model.joblib','stacking_selected_settings.json','test_evaluation_record.json']]
        if any(p.exists() for p in targets):
            raise FileExistsError('Meta checkpoint or settings already exist.')


    if stage.startswith('folds_'):
        seeds = [42] if stage=='folds_42' else [43,44,45,46]
        targets = [STACKING/f'seed_{s}'/name for s in seeds
                   for name in ['PRIVATE_training_user_fold_plan.csv','fold_summary.csv','experiment_plan.json']]
    elif stage.startswith(('rf_','textcnn_','bertweet_')):
        family,group = stage.split('_',1)
        name = {'rf':'RF','textcnn':'TextCNN','bertweet':'BERTweet'}[family]
        seeds = [42] if group=='42' else [43,44,45,46]
        targets = [p for s in seeds for p in (STACKING/f'seed_{s}').glob(f'*{name}*')]
    elif stage=='transformer':
        targets = [p for s in [42,43,44,45,46] for p in (MULTISEED/f'seed_{s}').glob('*TFM_*')]
        targets += [MULTISEED/name for name in ['Transformer_multiseed_results.csv','Transformer_multiseed_mean_SD.csv',
                                              'All_models_multiseed_results.csv','All_models_multiseed_mean_SD.csv']]
    else:
        targets = []
    if any(p.exists() for p in targets):
        raise FileExistsError('Stage outputs already exist.')

def partition(data, seed):
    folder = MULTISEED / f'seed_{seed}'
    require_files([folder/f'{part}_users.csv' for part in ['train','val','test']])
    parts = {}
    for part in ['train','val','test']:
        names = pd.read_csv(folder/f'{part}_users.csv')['username']
        assert names.notna().all() and names.is_unique
        parts[part] = set(names)
    assert tuple(len(parts[p]) for p in ['train','val','test']) == (146,37,61)
    assert not (parts['train'] & parts['val'] or parts['train'] & parts['test'] or parts['val'] & parts['test'])
    assert set().union(*parts.values()) == set(data['username'])
    positions = {p:np.flatnonzero(data['username'].isin(names).to_numpy()) for p,names in parts.items()}
    labels = data.iloc[positions['test']].groupby('username',sort=True)['label'].first()
    assert labels.value_counts().to_dict() == {1:54,0:7}
    return parts, positions

def check_users(users, data, positions, prediction='prediction', probability='ASD_probability', vote='vote_fraction'):
    expected = data.iloc[positions].groupby('username',sort=True)['label'].first()
    assert users.index.is_unique and set(users.index) == set(expected.index)
    users = users.loc[expected.index].copy()
    assert np.array_equal(users['label'].to_numpy(), expected.to_numpy())
    assert users[prediction].isin([0,1]).all()
    assert np.isfinite(users[probability]).all() and users[probability].between(0,1).all()
    assert users[vote].between(0,1).all()
    assert np.array_equal(users[prediction].to_numpy(), (users[vote]>=0.5).astype(int).to_numpy())
    return users

def check_metric_row(row, values):
    for key,value in values.items():
        assert key in row.index and np.isclose(float(row[key]),value,atol=1e-8,rtol=0), key


In [2]:
ENABLE_PREPROCESSING = False
ENABLE_ANALYSIS = False
PREPARATION_STAGE = None
ANALYSIS_STAGE = None

def require_source_stage(stage, mode, outputs=(), inputs=()):
    choices = {
        'preprocessing': (ENABLE_PREPROCESSING, PREPARATION_STAGE),
        'analysis': (ENABLE_ANALYSIS, ANALYSIS_STAGE),
        'training': (ENABLE_TRAINING, TRAINING_STAGE)
    }
    enabled, selected = choices[mode]
    if enabled is not True or selected != stage:
        raise RuntimeError(f'{stage} is disabled. Select its stage and enable {mode} explicitly.')
    existing = []
    for pattern in outputs:
        if any(character in pattern for character in '*?['):
            existing.extend(str(path) for path in BASE.glob(pattern))
        elif (BASE/pattern).exists():
            existing.append(str(BASE/pattern))
    if existing:
        raise FileExistsError('Stage outputs already exist:\n'+'\n'.join(existing))
    require_files([BASE/path for path in inputs])


## 2. Dataset and Representations

Training-only balancing, TF-IDF, and tokenization with user-disjoint partitions.


In [3]:
if not (BASE/'02_preprocessing/cleaned_data.pkl').is_file():
    try:
        from google.colab import drive
    except ImportError:
        raise FileNotFoundError(f'Dataset path: {BASE}') from None
    drive.mount('/content/drive_research')


Mounted at /content/drive_research


In [4]:
require_files([BASE/'02_preprocessing/cleaned_data.pkl'])
data = pd.read_pickle(BASE/'02_preprocessing/cleaned_data.pkl')[
    ['username','label','cleaned_tweet']
].reset_index(drop=True)
assert data.notna().all().all()
assert data['label'].isin([0,1]).all()
assert data.groupby('username')['label'].nunique().eq(1).all()
assert data['username'].nunique() == 244
partitions = {seed:partition(data,seed) for seed in SEEDS}
display(pd.DataFrame([{'tweets':len(data),'users':data['username'].nunique()}]))


,tweets,users
0,920070,244


In [ ]:
REVISION = BASE / '17_conference_revision'
LINGUISTIC = REVISION / 'linguistic_robustness'
REPORT_BASE_MODELS = ['Logistic Regression', 'SVM', 'Random Forest', 'Bi-LSTM', 'TextCNN', 'DistilBERT', 'RoBERTa', 'BERTweet']

def read_report_csv(path, **kwargs):
    require_files([path])
    return pd.read_csv(path, **kwargs)

def show_result(title, function):
    print(title)
    try:
        table = function()
    except FileNotFoundError as error:
        print(str(error))
        return None
    display(table)
    return table


In [6]:
def prepare_balanced_holdout():
    require_source_stage('balanced_holdout', 'preprocessing', outputs=['04_balanced_data/train_df_balanced.pkl'], inputs=['02_preprocessing/cleaned_data.pkl', '03_split/train_users.csv', '03_split/val_users.csv', '03_split/test_users.csv'])
    import os, re, html, glob, random, pickle
    import numpy as np
    import pandas as pd
    from pathlib import Path
    SAVE_DIR = str(BASE)
    SEED = 42
    df = pd.read_pickle(BASE / '02_preprocessing/cleaned_data.pkl')
    random.seed(SEED)
    np.random.seed(SEED)
    from sklearn.model_selection import train_test_split
    SPLIT_DIR = os.path.join(SAVE_DIR, '03_split')
    os.makedirs(SPLIT_DIR, exist_ok=True)
    TRAIN_USERS_PATH = os.path.join(SPLIT_DIR, 'train_users.csv')
    VAL_USERS_PATH = os.path.join(SPLIT_DIR, 'val_users.csv')
    TEST_USERS_PATH = os.path.join(SPLIT_DIR, 'test_users.csv')
    if os.path.exists(TRAIN_USERS_PATH) and os.path.exists(VAL_USERS_PATH) and os.path.exists(TEST_USERS_PATH):
        train_users = pd.read_csv(TRAIN_USERS_PATH)
        val_users = pd.read_csv(VAL_USERS_PATH)
        test_users = pd.read_csv(TEST_USERS_PATH)
    else:
        user_table = df.groupby('username', as_index=False)['label'].first()
        train_pool_users, test_users = train_test_split(user_table, test_size=0.25, random_state=SEED, stratify=user_table['label'])
        train_users, val_users = train_test_split(train_pool_users, test_size=0.2, random_state=SEED, stratify=train_pool_users['label'])
        train_users.to_csv(TRAIN_USERS_PATH, index=False)
        val_users.to_csv(VAL_USERS_PATH, index=False)
        test_users.to_csv(TEST_USERS_PATH, index=False)
    train_df_full = df[df['username'].isin(train_users['username'])].copy()
    val_df = df[df['username'].isin(val_users['username'])].copy()
    test_df = df[df['username'].isin(test_users['username'])].copy()
    for name, part in [('Train', train_df_full), ('Val', val_df), ('Test', test_df)]:
        pass
    train_set = set(train_df_full['username'])
    val_set = set(val_df['username'])
    test_set = set(test_df['username'])
    assert train_set.isdisjoint(val_set)
    assert train_set.isdisjoint(test_set)
    assert val_set.isdisjoint(test_set)
    all_users = set(df['username'])
    split_users = train_set | val_set | test_set
    assert all_users == split_users
    BALANCE_DIR = os.path.join(SAVE_DIR, '04_balanced_data')
    os.makedirs(BALANCE_DIR, exist_ok=True)
    TRAIN_BALANCED_PATH = os.path.join(BALANCE_DIR, 'train_df_balanced.pkl')
    if os.path.exists(TRAIN_BALANCED_PATH):
        train_df = pd.read_pickle(TRAIN_BALANCED_PATH)
    else:
        class_counts = train_df_full['label'].value_counts()
        minority_n = class_counts.min()
        train_parts = []
        for label_value, group in train_df_full.groupby('label'):
            if len(group) > minority_n:
                group = group.sample(n=minority_n, random_state=SEED)
            train_parts.append(group)
        train_df = pd.concat(train_parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
        train_df.to_pickle(TRAIN_BALANCED_PATH)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    return pd.DataFrame([{'class': label, 'tweets': int(n)} for label, n in train_df['label'].value_counts().items()])


In [7]:
stage_functions = {
    'balanced_holdout': prepare_balanced_holdout
}
for selected, enabled in [(PREPARATION_STAGE,ENABLE_PREPROCESSING),(ANALYSIS_STAGE,ENABLE_ANALYSIS),(TRAINING_STAGE,ENABLE_TRAINING)]:
    if enabled is True and selected in stage_functions:
        stage_result = stage_functions[selected]()
        if stage_result is not None:
            display(stage_result)


## 3. ML Models

Logistic Regression, calibrated LinearSVC, and Random Forest. The holdout and five-seed protocols have separate artifacts.


In [8]:
def train_holdout_ml():
    require_source_stage('holdout_ml', 'training', outputs=['07_ml_models/*', '08_ml_validation/*', '06_tfidf/*'], inputs=['02_preprocessing/cleaned_data.pkl', '03_split/train_users.csv', '03_split/val_users.csv', '03_split/test_users.csv'])
    import os, re, html, glob, random, pickle
    import numpy as np
    import pandas as pd
    from pathlib import Path
    SAVE_DIR = str(BASE)
    SEED = 42
    df = pd.read_pickle(BASE / '02_preprocessing/cleaned_data.pkl')
    random.seed(SEED)
    np.random.seed(SEED)
    from sklearn.model_selection import train_test_split
    SPLIT_DIR = os.path.join(SAVE_DIR, '03_split')
    os.makedirs(SPLIT_DIR, exist_ok=True)
    TRAIN_USERS_PATH = os.path.join(SPLIT_DIR, 'train_users.csv')
    VAL_USERS_PATH = os.path.join(SPLIT_DIR, 'val_users.csv')
    TEST_USERS_PATH = os.path.join(SPLIT_DIR, 'test_users.csv')
    if os.path.exists(TRAIN_USERS_PATH) and os.path.exists(VAL_USERS_PATH) and os.path.exists(TEST_USERS_PATH):
        train_users = pd.read_csv(TRAIN_USERS_PATH)
        val_users = pd.read_csv(VAL_USERS_PATH)
        test_users = pd.read_csv(TEST_USERS_PATH)
    else:
        user_table = df.groupby('username', as_index=False)['label'].first()
        train_pool_users, test_users = train_test_split(user_table, test_size=0.25, random_state=SEED, stratify=user_table['label'])
        train_users, val_users = train_test_split(train_pool_users, test_size=0.2, random_state=SEED, stratify=train_pool_users['label'])
        train_users.to_csv(TRAIN_USERS_PATH, index=False)
        val_users.to_csv(VAL_USERS_PATH, index=False)
        test_users.to_csv(TEST_USERS_PATH, index=False)
    train_df_full = df[df['username'].isin(train_users['username'])].copy()
    val_df = df[df['username'].isin(val_users['username'])].copy()
    test_df = df[df['username'].isin(test_users['username'])].copy()
    for name, part in [('Train', train_df_full), ('Val', val_df), ('Test', test_df)]:
        pass
    train_set = set(train_df_full['username'])
    val_set = set(val_df['username'])
    test_set = set(test_df['username'])
    assert train_set.isdisjoint(val_set)
    assert train_set.isdisjoint(test_set)
    assert val_set.isdisjoint(test_set)
    all_users = set(df['username'])
    split_users = train_set | val_set | test_set
    assert all_users == split_users
    BALANCE_DIR = os.path.join(SAVE_DIR, '04_balanced_data')
    os.makedirs(BALANCE_DIR, exist_ok=True)
    TRAIN_BALANCED_PATH = os.path.join(BALANCE_DIR, 'train_df_balanced.pkl')
    if os.path.exists(TRAIN_BALANCED_PATH):
        train_df = pd.read_pickle(TRAIN_BALANCED_PATH)
    else:
        class_counts = train_df_full['label'].value_counts()
        minority_n = class_counts.min()
        train_parts = []
        for label_value, group in train_df_full.groupby('label'):
            if len(group) > minority_n:
                group = group.sample(n=minority_n, random_state=SEED)
            train_parts.append(group)
        train_df = pd.concat(train_parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
        train_df.to_pickle(TRAIN_BALANCED_PATH)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    import os
    import numpy as np
    import pandas as pd
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

    def metric_dict(y_true, y_pred, y_prob=None):
        result = {'accuracy': accuracy_score(y_true, y_pred), 'precision': precision_score(y_true, y_pred, zero_division=0), 'recall': recall_score(y_true, y_pred, zero_division=0), 'f1': f1_score(y_true, y_pred, zero_division=0), 'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0)}
        if y_prob is not None:
            try:
                result['roc_auc'] = roc_auc_score(y_true, y_prob)
            except:
                result['roc_auc'] = np.nan
        return result

    def evaluate_tweet_and_user(data_df, pred, prob, model_name, save=True):
        y_true = data_df['label'].values
        tweet_metrics = metric_dict(y_true, pred, prob)
        tweet_result = {'model': model_name, 'level': 'tweet', **tweet_metrics}
        user_eval = data_df[['username', 'label']].copy()
        user_eval['pred'] = pred
        user_eval['prob'] = prob
        user_eval = user_eval.groupby('username').agg(label=('label', 'first'), pred=('pred', 'mean'), prob=('prob', 'mean')).reset_index()
        user_eval['pred'] = (user_eval['pred'] >= 0.5).astype(int)
        user_metrics = metric_dict(user_eval['label'].values, user_eval['pred'].values, user_eval['prob'].values)
        user_result = {'model': model_name, 'level': 'user', **user_metrics}
        if save:
            if 'all_results' not in locals():
                locals()['all_results'] = []
            all_results.append(tweet_result)
            all_results.append(user_result)
        return (tweet_result, user_result, tweet_metrics, user_eval)
    from sklearn.feature_extraction.text import TfidfVectorizer
    import joblib
    TFIDF_DIR = os.path.join(SAVE_DIR, '06_tfidf')
    os.makedirs(TFIDF_DIR, exist_ok=True)
    TFIDF_MODEL_PATH = os.path.join(TFIDF_DIR, 'tfidf_vectorizer.pkl')
    X_TRAIN_TFIDF_PATH = os.path.join(TFIDF_DIR, 'X_train_tfidf.npz')
    X_VAL_TFIDF_PATH = os.path.join(TFIDF_DIR, 'X_val_tfidf.npz')
    X_TEST_TFIDF_PATH = os.path.join(TFIDF_DIR, 'X_test_tfidf.npz')
    Y_TRAIN_PATH = os.path.join(TFIDF_DIR, 'y_train.npy')
    Y_VAL_PATH = os.path.join(TFIDF_DIR, 'y_val.npy')
    Y_TEST_PATH = os.path.join(TFIDF_DIR, 'y_test.npy')
    tfidf_files_exist = all((os.path.exists(path) for path in [TFIDF_MODEL_PATH, X_TRAIN_TFIDF_PATH, X_VAL_TFIDF_PATH, X_TEST_TFIDF_PATH, Y_TRAIN_PATH, Y_VAL_PATH, Y_TEST_PATH]))
    if tfidf_files_exist:
        tfidf = joblib.load(TFIDF_MODEL_PATH)
        X_train_tfidf = __import__('scipy.sparse', fromlist=['load_npz']).load_npz(X_TRAIN_TFIDF_PATH)
        X_val_tfidf = __import__('scipy.sparse', fromlist=['load_npz']).load_npz(X_VAL_TFIDF_PATH)
        X_test_tfidf = __import__('scipy.sparse', fromlist=['load_npz']).load_npz(X_TEST_TFIDF_PATH)
        y_train = np.load(Y_TRAIN_PATH)
        y_val = np.load(Y_VAL_PATH)
        y_test = np.load(Y_TEST_PATH)
    else:
        tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
        X_train_tfidf = tfidf.fit_transform(train_df['cleaned_tweet'])
        X_val_tfidf = tfidf.transform(val_df['cleaned_tweet'])
        X_test_tfidf = tfidf.transform(test_df['cleaned_tweet'])
        y_train = train_df['label'].to_numpy()
        y_val = val_df['label'].to_numpy()
        y_test = test_df['label'].to_numpy()
        joblib.dump(tfidf, TFIDF_MODEL_PATH)
        from scipy.sparse import save_npz
        save_npz(X_TRAIN_TFIDF_PATH, X_train_tfidf)
        save_npz(X_VAL_TFIDF_PATH, X_val_tfidf)
        save_npz(X_TEST_TFIDF_PATH, X_test_tfidf)
        np.save(Y_TRAIN_PATH, y_train)
        np.save(Y_VAL_PATH, y_val)
        np.save(Y_TEST_PATH, y_test)
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import LinearSVC
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.ensemble import RandomForestClassifier
    import joblib
    ML_MODEL_DIR = os.path.join(SAVE_DIR, '07_ml_models')
    os.makedirs(ML_MODEL_DIR, exist_ok=True)
    LR_PATH = os.path.join(ML_MODEL_DIR, 'logistic_regression.pkl')
    SVM_PATH = os.path.join(ML_MODEL_DIR, 'svm_calibrated.pkl')
    RF_PATH = os.path.join(ML_MODEL_DIR, 'random_forest.pkl')
    all_models_exist = all((os.path.exists(path) for path in [LR_PATH, SVM_PATH, RF_PATH]))
    if all_models_exist:
        lr = joblib.load(LR_PATH)
        svm_calibrated = joblib.load(SVM_PATH)
        rf = joblib.load(RF_PATH)
        ml_models = {'Logistic Regression': lr, 'SVM': svm_calibrated, 'Random Forest': rf}
    else:
        ml_models = {}
        lr = LogisticRegression(max_iter=1000, random_state=SEED)
        lr.fit(X_train_tfidf, y_train)
        ml_models['Logistic Regression'] = lr
        joblib.dump(lr, LR_PATH)
        base_svm = LinearSVC(random_state=SEED, max_iter=3000)
        svm_calibrated = CalibratedClassifierCV(base_svm, cv=3)
        svm_calibrated.fit(X_train_tfidf, y_train)
        ml_models['SVM'] = svm_calibrated
        joblib.dump(svm_calibrated, SVM_PATH)
        rf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
        rf.fit(X_train_tfidf, y_train)
        ml_models['Random Forest'] = rf
        joblib.dump(rf, RF_PATH)
    for name in ml_models:
        pass
    import joblib
    ML_VAL_DIR = os.path.join(SAVE_DIR, '08_ml_validation')
    os.makedirs(ML_VAL_DIR, exist_ok=True)
    ML_VAL_TABLE_PATH = os.path.join(ML_VAL_DIR, 'ml_val_results.csv')
    ML_VAL_PRED_PATH = os.path.join(ML_VAL_DIR, 'ml_val_predictions.pkl')
    BEST_ML_PATH = os.path.join(ML_VAL_DIR, 'best_ml_name.txt')
    val_checkpoint_exists = all((os.path.exists(path) for path in [ML_VAL_TABLE_PATH, ML_VAL_PRED_PATH, BEST_ML_PATH]))
    if val_checkpoint_exists:
        ml_val_table = pd.read_csv(ML_VAL_TABLE_PATH)
        ml_val_predictions = joblib.load(ML_VAL_PRED_PATH)
        with open(BEST_ML_PATH, 'r') as f:
            best_ml_name = f.read().strip()
    else:
        ml_val_results = []
        ml_val_predictions = {}
        for name, model in ml_models.items():
            val_pred = model.predict(X_val_tfidf)
            val_prob = model.predict_proba(X_val_tfidf)[:, 1]
            tweet_result, user_result, _, _ = evaluate_tweet_and_user(val_df, val_pred, val_prob, name, save=False)
            ml_val_results.append({**tweet_result, 'user_macro_f1': user_result['macro_f1'], 'user_accuracy': user_result['accuracy'], 'user_roc_auc': user_result['roc_auc']})
            ml_val_predictions[name] = {'pred': val_pred, 'prob': val_prob}
        ml_val_table = pd.DataFrame(ml_val_results)
        best_ml_name = ml_val_table.sort_values('macro_f1', ascending=False).iloc[0]['model']
        ml_val_table.to_csv(ML_VAL_TABLE_PATH, index=False)
        joblib.dump(ml_val_predictions, ML_VAL_PRED_PATH)
        with open(BEST_ML_PATH, 'w') as f:
            f.write(best_ml_name)
    display(ml_val_table.sort_values('macro_f1', ascending=False).round(4))
    all_results = []
    test_prediction_store = {}

    def add(name, pred, prob):
        assert len(pred) == len(test_df)
        evaluate_tweet_and_user(test_df, pred, prob, name, save=True)
        test_prediction_store[name] = {'pred': pred, 'prob': prob}
    for name, model in ml_models.items():
        add(name, model.predict(X_test_tfidf), model.predict_proba(X_test_tfidf)[:, 1])
    return pd.DataFrame(all_results)


In [9]:
def train_multiseed_ml():
    require_source_stage('standalone_ml', 'training', outputs=['17_conference_revision/multiseed/ML_*', '17_conference_revision/multiseed/seed_*/ML_*', '17_conference_revision/multiseed/seed_*/PRIVATE_ML_*'], inputs=['02_preprocessing/cleaned_data.pkl', '04_balanced_data/train_df_balanced.pkl', '17_conference_revision/multiseed/seed_42/train_users.csv', '17_conference_revision/multiseed/seed_42/val_users.csv', '17_conference_revision/multiseed/seed_42/test_users.csv', '17_conference_revision/multiseed/seed_43/train_users.csv', '17_conference_revision/multiseed/seed_43/val_users.csv', '17_conference_revision/multiseed/seed_43/test_users.csv', '17_conference_revision/multiseed/seed_44/train_users.csv', '17_conference_revision/multiseed/seed_44/val_users.csv', '17_conference_revision/multiseed/seed_44/test_users.csv', '17_conference_revision/multiseed/seed_45/train_users.csv', '17_conference_revision/multiseed/seed_45/val_users.csv', '17_conference_revision/multiseed/seed_45/test_users.csv', '17_conference_revision/multiseed/seed_46/train_users.csv', '17_conference_revision/multiseed/seed_46/val_users.csv', '17_conference_revision/multiseed/seed_46/test_users.csv'])
    from pathlib import Path
    import gc
    import json
    import hashlib
    import platform
    import importlib.metadata as metadata
    import numpy as np
    import pandas as pd
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import LinearSVC
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
    BASE = Path('/content/drive_research/MyDrive/thesis/saved_results')
    ROOT = BASE / '17_conference_revision/multiseed'
    data = pd.read_pickle(BASE / '02_preprocessing/cleaned_data.pkl')[['username', 'label', 'cleaned_tweet']].reset_index(drop=True)
    assert data['cleaned_tweet'].notna().all()
    assert data.groupby('username')['label'].nunique().eq(1).all()
    versions = {'python': platform.python_version()}
    for package in ['numpy', 'pandas', 'scipy', 'scikit-learn']:
        versions[package] = metadata.version(package)
    (ROOT / 'ML_software_versions.json').write_text(json.dumps(versions, indent=2))
    data_hash = hashlib.sha256(pd.util.hash_pandas_object(data, index=True).values.tobytes()).hexdigest()

    def user_metrics(test_frame, pred, prob):
        temp = test_frame[['username', 'label']].copy()
        temp['pred'] = pred
        temp['prob'] = prob
        user = temp.groupby('username', sort=True).agg(label=('label', 'first'), vote=('pred', 'mean'), prob=('prob', 'mean'))
        user['pred'] = (user['vote'] >= 0.5).astype(int)
        y = user['label']
        p = user['pred']
        scores = {'accuracy': accuracy_score(y, p), 'precision_ASD': precision_score(y, p, zero_division=0), 'recall_ASD': recall_score(y, p, zero_division=0), 'f1_ASD': f1_score(y, p, zero_division=0), 'macro_f1': f1_score(y, p, average='macro', labels=[0, 1], zero_division=0), 'recall_non_ASD': recall_score(y, p, pos_label=0, zero_division=0), 'f1_non_ASD': f1_score(y, p, pos_label=0, zero_division=0), 'roc_auc': roc_auc_score(y, user['prob'])}
        return (scores, user)
    for seed in [42, 43, 44, 45, 46]:
        folder = ROOT / f'seed_{seed}'
        parts = {part: pd.read_csv(folder / f'{part}_users.csv') for part in ['train', 'val', 'test']}
        train_names = set(parts['train']['username'])
        val_names = set(parts['val']['username'])
        test_names = set(parts['test']['username'])
        assert not (train_names & val_names or train_names & test_names or val_names & test_names)
        train_full = data[data['username'].isin(train_names)].copy()
        test_frame = data[data['username'].isin(test_names)].reset_index(drop=True)
        if seed == 42:
            train_balanced = pd.read_pickle(BASE / '04_balanced_data/train_df_balanced.pkl')[['username', 'label', 'cleaned_tweet']].copy()
            assert set(train_balanced['username']).issubset(train_names)
        else:
            n_minority = train_full['label'].value_counts().min()
            pieces = []
            for label, group in train_full.groupby('label'):
                if len(group) > n_minority:
                    group = group.sample(n=n_minority, random_state=seed)
                pieces.append(group)
            train_balanced = pd.concat(pieces).sample(frac=1, random_state=seed).reset_index(drop=True)
        assert train_balanced['label'].nunique() == 2
        assert train_balanced['label'].value_counts().nunique() == 1
        models = {'Logistic Regression': LogisticRegression(max_iter=1000, random_state=seed), 'SVM': CalibratedClassifierCV(estimator=LinearSVC(max_iter=3000, random_state=seed), method='sigmoid', cv=3), 'Random Forest': RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)}
        vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
        X_train = vectorizer.fit_transform(train_balanced['cleaned_tweet'])
        X_test = vectorizer.transform(test_frame['cleaned_tweet'])
        for name, model in models.items():
            safe_name = name.replace(' ', '_')
            result_path = folder / f'ML_{safe_name}_metrics.csv'
            config_path = folder / f'ML_{safe_name}_config.json'
            config = {'protocol': 'ML_multiseed_v1', 'seed': seed, 'data_hash': data_hash, 'versions': versions, 'model_parameters': model.get_params(deep=True), 'vectorizer_parameters': vectorizer.get_params(), 'train_tweets': len(train_balanced), 'test_tweets': len(test_frame), 'train_users': sorted(train_names), 'test_users': sorted(test_names)}
            signature = hashlib.sha256(json.dumps(config, sort_keys=True, default=str).encode()).hexdigest()
            if result_path.exists() and config_path.exists():
                previous = json.loads(config_path.read_text())
                assert previous['signature'] == signature, 'Artifact validation failed.'
                continue
            model.fit(X_train, train_balanced['label'])
            pred = model.predict(X_test)
            prob = model.predict_proba(X_test)[:, 1]
            scores, user = user_metrics(test_frame, pred, prob)
            config['signature'] = signature
            config_path.write_text(json.dumps(config, indent=2, default=str))
            user.to_csv(folder / f'PRIVATE_ML_{safe_name}_user_predictions.csv')
            pd.DataFrame([{'model': name, 'seed': seed, **scores}]).to_csv(result_path, index=False)
        del X_train, X_test, models, vectorizer
        del train_full, train_balanced, test_frame
        gc.collect()
    files = sorted(ROOT.glob('seed_*/ML_*_metrics.csv'))
    results = pd.concat([pd.read_csv(path) for path in files], ignore_index=True)
    assert len(results) == 15
    assert not results.duplicated(['model', 'seed']).any()
    results.to_csv(ROOT / 'ML_multiseed_results.csv', index=False)
    metric_columns = ['accuracy', 'precision_ASD', 'recall_ASD', 'f1_ASD', 'macro_f1', 'recall_non_ASD', 'f1_non_ASD', 'roc_auc']
    summary = results.groupby('model')[metric_columns].agg(['mean', 'std'])
    summary.to_csv(ROOT / 'ML_multiseed_mean_SD.csv')
    display(results[['model', 'seed', 'macro_f1', 'roc_auc']])
    display(summary[['macro_f1', 'roc_auc']].round(2))
    return results


In [10]:
stage_functions = {
    'holdout_ml': train_holdout_ml,
    'standalone_ml': train_multiseed_ml
}
for selected, enabled in [(PREPARATION_STAGE,ENABLE_PREPROCESSING),(ANALYSIS_STAGE,ENABLE_ANALYSIS),(TRAINING_STAGE,ENABLE_TRAINING)]:
    if enabled is True and selected in stage_functions:
        stage_result = stage_functions[selected]()
        if stage_result is not None:
            display(stage_result)


## 4. DL Models

Bi-LSTM and TextCNN: vocabulary 10,000, length 100, embedding 64. Early stopping uses validation loss; patience 3, maximum 10 epochs.


In [11]:
def train_holdout_dl():
    require_source_stage('holdout_dl', 'training', outputs=['09_tokenizer/*', '10_dl_models/bilstm/*', '10_dl_models/textcnn/*'], inputs=['02_preprocessing/cleaned_data.pkl', '03_split/train_users.csv', '03_split/val_users.csv', '03_split/test_users.csv'])
    import os, re, html, glob, random, pickle
    import numpy as np
    import pandas as pd
    from pathlib import Path
    SAVE_DIR = str(BASE)
    SEED = 42
    df = pd.read_pickle(BASE / '02_preprocessing/cleaned_data.pkl')
    random.seed(SEED)
    np.random.seed(SEED)
    import tensorflow as tf
    tf.random.set_seed(SEED)
    from sklearn.model_selection import train_test_split
    SPLIT_DIR = os.path.join(SAVE_DIR, '03_split')
    os.makedirs(SPLIT_DIR, exist_ok=True)
    TRAIN_USERS_PATH = os.path.join(SPLIT_DIR, 'train_users.csv')
    VAL_USERS_PATH = os.path.join(SPLIT_DIR, 'val_users.csv')
    TEST_USERS_PATH = os.path.join(SPLIT_DIR, 'test_users.csv')
    if os.path.exists(TRAIN_USERS_PATH) and os.path.exists(VAL_USERS_PATH) and os.path.exists(TEST_USERS_PATH):
        train_users = pd.read_csv(TRAIN_USERS_PATH)
        val_users = pd.read_csv(VAL_USERS_PATH)
        test_users = pd.read_csv(TEST_USERS_PATH)
    else:
        user_table = df.groupby('username', as_index=False)['label'].first()
        train_pool_users, test_users = train_test_split(user_table, test_size=0.25, random_state=SEED, stratify=user_table['label'])
        train_users, val_users = train_test_split(train_pool_users, test_size=0.2, random_state=SEED, stratify=train_pool_users['label'])
        train_users.to_csv(TRAIN_USERS_PATH, index=False)
        val_users.to_csv(VAL_USERS_PATH, index=False)
        test_users.to_csv(TEST_USERS_PATH, index=False)
    train_df_full = df[df['username'].isin(train_users['username'])].copy()
    val_df = df[df['username'].isin(val_users['username'])].copy()
    test_df = df[df['username'].isin(test_users['username'])].copy()
    for name, part in [('Train', train_df_full), ('Val', val_df), ('Test', test_df)]:
        pass
    train_set = set(train_df_full['username'])
    val_set = set(val_df['username'])
    test_set = set(test_df['username'])
    assert train_set.isdisjoint(val_set)
    assert train_set.isdisjoint(test_set)
    assert val_set.isdisjoint(test_set)
    all_users = set(df['username'])
    split_users = train_set | val_set | test_set
    assert all_users == split_users
    BALANCE_DIR = os.path.join(SAVE_DIR, '04_balanced_data')
    os.makedirs(BALANCE_DIR, exist_ok=True)
    TRAIN_BALANCED_PATH = os.path.join(BALANCE_DIR, 'train_df_balanced.pkl')
    if os.path.exists(TRAIN_BALANCED_PATH):
        train_df = pd.read_pickle(TRAIN_BALANCED_PATH)
    else:
        class_counts = train_df_full['label'].value_counts()
        minority_n = class_counts.min()
        train_parts = []
        for label_value, group in train_df_full.groupby('label'):
            if len(group) > minority_n:
                group = group.sample(n=minority_n, random_state=SEED)
            train_parts.append(group)
        train_df = pd.concat(train_parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
        train_df.to_pickle(TRAIN_BALANCED_PATH)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    import os
    import numpy as np
    import pandas as pd
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

    def metric_dict(y_true, y_pred, y_prob=None):
        result = {'accuracy': accuracy_score(y_true, y_pred), 'precision': precision_score(y_true, y_pred, zero_division=0), 'recall': recall_score(y_true, y_pred, zero_division=0), 'f1': f1_score(y_true, y_pred, zero_division=0), 'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0)}
        if y_prob is not None:
            try:
                result['roc_auc'] = roc_auc_score(y_true, y_prob)
            except:
                result['roc_auc'] = np.nan
        return result

    def evaluate_tweet_and_user(data_df, pred, prob, model_name, save=True):
        y_true = data_df['label'].values
        tweet_metrics = metric_dict(y_true, pred, prob)
        tweet_result = {'model': model_name, 'level': 'tweet', **tweet_metrics}
        user_eval = data_df[['username', 'label']].copy()
        user_eval['pred'] = pred
        user_eval['prob'] = prob
        user_eval = user_eval.groupby('username').agg(label=('label', 'first'), pred=('pred', 'mean'), prob=('prob', 'mean')).reset_index()
        user_eval['pred'] = (user_eval['pred'] >= 0.5).astype(int)
        user_metrics = metric_dict(user_eval['label'].values, user_eval['pred'].values, user_eval['prob'].values)
        user_result = {'model': model_name, 'level': 'user', **user_metrics}
        if save:
            if 'all_results' not in locals():
                locals()['all_results'] = []
            all_results.append(tweet_result)
            all_results.append(user_result)
        return (tweet_result, user_result, tweet_metrics, user_eval)
    y_train = train_df['label'].to_numpy()
    y_val = val_df['label'].to_numpy()
    y_test = test_df['label'].to_numpy()
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    import pickle
    MAX_WORDS = 10000
    MAX_LEN = 100
    EMBED_DIM = 64
    TOKENIZER_DIR = os.path.join(SAVE_DIR, '09_tokenizer')
    os.makedirs(TOKENIZER_DIR, exist_ok=True)
    TOKENIZER_PATH = os.path.join(TOKENIZER_DIR, 'keras_tokenizer.pkl')
    X_TRAIN_SEQ_PATH = os.path.join(TOKENIZER_DIR, 'X_train_seq.npy')
    X_VAL_SEQ_PATH = os.path.join(TOKENIZER_DIR, 'X_val_seq.npy')
    X_TEST_SEQ_PATH = os.path.join(TOKENIZER_DIR, 'X_test_seq.npy')
    PARAMS_PATH = os.path.join(TOKENIZER_DIR, 'tokenizer_params.pkl')
    tokenizer_files_exist = all((os.path.exists(path) for path in [TOKENIZER_PATH, X_TRAIN_SEQ_PATH, X_VAL_SEQ_PATH, X_TEST_SEQ_PATH, PARAMS_PATH]))
    if tokenizer_files_exist:
        with open(TOKENIZER_PATH, 'rb') as f:
            keras_tokenizer = pickle.load(f)
        X_train_seq = np.load(X_TRAIN_SEQ_PATH)
        X_val_seq = np.load(X_VAL_SEQ_PATH)
        X_test_seq = np.load(X_TEST_SEQ_PATH)
        with open(PARAMS_PATH, 'rb') as f:
            tokenizer_params = pickle.load(f)
        MAX_WORDS = tokenizer_params['MAX_WORDS']
        MAX_LEN = tokenizer_params['MAX_LEN']
        EMBED_DIM = tokenizer_params['EMBED_DIM']
    else:
        keras_tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
        keras_tokenizer.fit_on_texts(train_df['cleaned_tweet'])

        def keras_encode(text_series):
            seq = keras_tokenizer.texts_to_sequences(text_series)
            return pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
        X_train_seq = keras_encode(train_df['cleaned_tweet'])
        X_val_seq = keras_encode(val_df['cleaned_tweet'])
        X_test_seq = keras_encode(test_df['cleaned_tweet'])
        with open(TOKENIZER_PATH, 'wb') as f:
            pickle.dump(keras_tokenizer, f)
        np.save(X_TRAIN_SEQ_PATH, X_train_seq)
        np.save(X_VAL_SEQ_PATH, X_val_seq)
        np.save(X_TEST_SEQ_PATH, X_test_seq)
        tokenizer_params = {'MAX_WORDS': MAX_WORDS, 'MAX_LEN': MAX_LEN, 'EMBED_DIM': EMBED_DIM}
        with open(PARAMS_PATH, 'wb') as f:
            pickle.dump(tokenizer_params, f)
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, Conv1D, GlobalMaxPooling1D
    from tensorflow.keras.callbacks import EarlyStopping

    def build_bilstm():
        model = Sequential([Embedding(input_dim=MAX_WORDS, output_dim=EMBED_DIM, input_length=MAX_LEN), Bidirectional(LSTM(64)), Dropout(0.3), Dense(32, activation='relu'), Dropout(0.2), Dense(1, activation='sigmoid')])
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        return model

    def build_textcnn():
        model = Sequential([Embedding(input_dim=MAX_WORDS, output_dim=EMBED_DIM, input_length=MAX_LEN), Conv1D(filters=128, kernel_size=3, activation='relu'), GlobalMaxPooling1D(), Dropout(0.3), Dense(64, activation='relu'), Dropout(0.2), Dense(1, activation='sigmoid')])
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        return model
    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    import os
    import pickle
    import numpy as np
    from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
    from tensorflow.keras.models import load_model
    BILSTM_DIR = os.path.join(SAVE_DIR, '10_dl_models', 'bilstm')
    os.makedirs(BILSTM_DIR, exist_ok=True)
    BILSTM_MODEL_PATH = os.path.join(BILSTM_DIR, 'best_model.keras')
    BILSTM_HISTORY_PATH = os.path.join(BILSTM_DIR, 'history.pkl')
    BILSTM_VAL_PROB_PATH = os.path.join(BILSTM_DIR, 'val_prob.npy')
    BILSTM_VAL_PRED_PATH = os.path.join(BILSTM_DIR, 'val_pred.npy')
    bilstm_checkpoint_exist = all((os.path.exists(path) for path in [BILSTM_MODEL_PATH, BILSTM_HISTORY_PATH, BILSTM_VAL_PROB_PATH, BILSTM_VAL_PRED_PATH]))
    if bilstm_checkpoint_exist:
        bilstm = load_model(BILSTM_MODEL_PATH)
        with open(BILSTM_HISTORY_PATH, 'rb') as f:
            history_bilstm_data = pickle.load(f)
        history_bilstm = type('History', (), {'history': history_bilstm_data})()
        bilstm_val_prob = np.load(BILSTM_VAL_PROB_PATH)
        bilstm_val_pred = np.load(BILSTM_VAL_PRED_PATH)
    else:
        bilstm = build_bilstm()
        bilstm_checkpoint = ModelCheckpoint(filepath=BILSTM_MODEL_PATH, monitor='val_loss', save_best_only=True, save_weights_only=False, verbose=1)
        early_stop_bilstm = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)
        history_bilstm = bilstm.fit(X_train_seq, y_train, validation_data=(X_val_seq, y_val), epochs=10, batch_size=64, callbacks=[bilstm_checkpoint, early_stop_bilstm], verbose=1)
        with open(BILSTM_HISTORY_PATH, 'wb') as f:
            pickle.dump(history_bilstm.history, f)
        bilstm = load_model(BILSTM_MODEL_PATH)
        bilstm_val_prob = bilstm.predict(X_val_seq, batch_size=256, verbose=1).ravel()
        bilstm_val_pred = (bilstm_val_prob >= 0.5).astype(int)
        np.save(BILSTM_VAL_PROB_PATH, bilstm_val_prob)
        np.save(BILSTM_VAL_PRED_PATH, bilstm_val_pred)
    bilstm_val_tweet, bilstm_val_user, _, _ = evaluate_tweet_and_user(val_df, bilstm_val_pred, bilstm_val_prob, 'Bi-LSTM', save=False)
    import os
    import pickle
    import numpy as np
    from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
    from tensorflow.keras.models import load_model
    TEXTCNN_DIR = os.path.join(SAVE_DIR, '10_dl_models', 'textcnn')
    os.makedirs(TEXTCNN_DIR, exist_ok=True)
    TEXTCNN_MODEL_PATH = os.path.join(TEXTCNN_DIR, 'best_model.keras')
    TEXTCNN_HISTORY_PATH = os.path.join(TEXTCNN_DIR, 'history.pkl')
    TEXTCNN_VAL_PROB_PATH = os.path.join(TEXTCNN_DIR, 'val_prob.npy')
    TEXTCNN_VAL_PRED_PATH = os.path.join(TEXTCNN_DIR, 'val_pred.npy')
    textcnn_checkpoint_exist = all((os.path.exists(path) for path in [TEXTCNN_MODEL_PATH, TEXTCNN_HISTORY_PATH, TEXTCNN_VAL_PROB_PATH, TEXTCNN_VAL_PRED_PATH]))
    if textcnn_checkpoint_exist:
        textcnn = load_model(TEXTCNN_MODEL_PATH)
        with open(TEXTCNN_HISTORY_PATH, 'rb') as f:
            history_textcnn_data = pickle.load(f)
        history_textcnn = type('History', (), {'history': history_textcnn_data})()
        textcnn_val_prob = np.load(TEXTCNN_VAL_PROB_PATH)
        textcnn_val_pred = np.load(TEXTCNN_VAL_PRED_PATH)
    else:
        textcnn = build_textcnn()
        textcnn_checkpoint = ModelCheckpoint(filepath=TEXTCNN_MODEL_PATH, monitor='val_loss', save_best_only=True, save_weights_only=False, verbose=1)
        early_stop_textcnn = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)
        history_textcnn = textcnn.fit(X_train_seq, y_train, validation_data=(X_val_seq, y_val), epochs=10, batch_size=64, callbacks=[textcnn_checkpoint, early_stop_textcnn], verbose=1)
        with open(TEXTCNN_HISTORY_PATH, 'wb') as f:
            pickle.dump(history_textcnn.history, f)
        textcnn = load_model(TEXTCNN_MODEL_PATH)
        textcnn_val_prob = textcnn.predict(X_val_seq, batch_size=256, verbose=1).ravel()
        textcnn_val_pred = (textcnn_val_prob >= 0.5).astype(int)
        np.save(TEXTCNN_VAL_PROB_PATH, textcnn_val_prob)
        np.save(TEXTCNN_VAL_PRED_PATH, textcnn_val_pred)
    textcnn_val_tweet, textcnn_val_user, _, _ = evaluate_tweet_and_user(val_df, textcnn_val_pred, textcnn_val_prob, 'TextCNN', save=False)
    all_results = []
    test_prediction_store = {}

    def add(name, pred, prob):
        assert len(pred) == len(test_df)
        evaluate_tweet_and_user(test_df, pred, prob, name, save=True)
        test_prediction_store[name] = {'pred': pred, 'prob': prob}
    for name, m in [('Bi-LSTM', bilstm), ('TextCNN', textcnn)]:
        prob = m.predict(X_test_seq, batch_size=256, verbose=0).ravel()
        add(name, (prob >= 0.5).astype(int), prob)
    return pd.DataFrame(all_results)


In [12]:
def train_multiseed_dl():
    require_source_stage('standalone_dl', 'training', outputs=['17_conference_revision/multiseed/DL_*', '17_conference_revision/multiseed/seed_*/DL_*', '17_conference_revision/multiseed/seed_*/PRIVATE_DL_*'], inputs=['02_preprocessing/cleaned_data.pkl', '04_balanced_data/train_df_balanced.pkl', '17_conference_revision/multiseed/seed_42/train_users.csv', '17_conference_revision/multiseed/seed_42/val_users.csv', '17_conference_revision/multiseed/seed_42/test_users.csv', '17_conference_revision/multiseed/seed_43/train_users.csv', '17_conference_revision/multiseed/seed_43/val_users.csv', '17_conference_revision/multiseed/seed_43/test_users.csv', '17_conference_revision/multiseed/seed_44/train_users.csv', '17_conference_revision/multiseed/seed_44/val_users.csv', '17_conference_revision/multiseed/seed_44/test_users.csv', '17_conference_revision/multiseed/seed_45/train_users.csv', '17_conference_revision/multiseed/seed_45/val_users.csv', '17_conference_revision/multiseed/seed_45/test_users.csv', '17_conference_revision/multiseed/seed_46/train_users.csv', '17_conference_revision/multiseed/seed_46/val_users.csv', '17_conference_revision/multiseed/seed_46/test_users.csv'])
    from pathlib import Path
    import gc
    import json
    import hashlib
    import platform
    import importlib.metadata as metadata
    import numpy as np
    import pandas as pd
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense, Dropout, Conv1D, GlobalMaxPooling1D
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.callbacks import EarlyStopping, CSVLogger, TerminateOnNaN
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
    from IPython.display import display
    gpus = tf.config.list_physical_devices('GPU')
    assert gpus, 'Artifact validation failed.'
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass
    tf.keras.mixed_precision.set_global_policy('float32')
    BASE = Path('/content/drive_research/MyDrive/thesis/saved_results')
    try:
        drive_ready = (BASE / '02_preprocessing/cleaned_data.pkl').is_file()
    except OSError:
        drive_ready = False
    if not drive_ready:
        pass
    assert (BASE / '02_preprocessing/cleaned_data.pkl').is_file(), 'Artifact validation failed.'
    ROOT = BASE / '17_conference_revision/multiseed'
    assert ROOT.is_dir(), 'Artifact validation failed.'
    dl_data = pd.read_pickle(BASE / '02_preprocessing/cleaned_data.pkl')[['username', 'label', 'cleaned_tweet']].reset_index(drop=True)
    assert dl_data['cleaned_tweet'].notna().all()
    assert dl_data['label'].isin([0, 1]).all()
    assert dl_data.groupby('username')['label'].nunique().eq(1).all()
    assert dl_data['username'].nunique() == 244
    SEEDS = [42, 43, 44, 45, 46]
    MAX_WORDS = 10000
    MAX_LEN = 100
    EMBED_DIM = 64
    MAX_EPOCHS = 10
    BATCH_SIZE = 64
    versions = {'python': platform.python_version(), 'tensorflow': tf.__version__}
    for package in ['numpy', 'pandas', 'scikit-learn', 'keras']:
        versions[package] = metadata.version(package)
    (ROOT / 'DL_software_versions.json').write_text(json.dumps(versions, indent=2))
    data_hash = hashlib.sha256(pd.util.hash_pandas_object(dl_data, index=True).values.tobytes()).hexdigest()

    def build_dl_model(name):
        if name == 'Bi-LSTM':
            model = Sequential([Input(shape=(MAX_LEN,), dtype='int32'), Embedding(MAX_WORDS, EMBED_DIM), Bidirectional(LSTM(64)), Dropout(0.3), Dense(32, activation='relu'), Dropout(0.2), Dense(1, activation='sigmoid')])
        else:
            model = Sequential([Input(shape=(MAX_LEN,), dtype='int32'), Embedding(MAX_WORDS, EMBED_DIM), Conv1D(filters=128, kernel_size=3, activation='relu'), GlobalMaxPooling1D(), Dropout(0.3), Dense(64, activation='relu'), Dropout(0.2), Dense(1, activation='sigmoid')])
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
        return model

    def evaluate_dl_users(frame, probability):
        probability = np.asarray(probability).reshape(-1)
        assert len(probability) == len(frame)
        assert np.isfinite(probability).all()
        assert ((probability >= 0) & (probability <= 1)).all()
        temp = frame[['username', 'label']].copy()
        temp['pred'] = (probability >= 0.5).astype(int)
        temp['prob'] = probability
        user = temp.groupby('username', sort=True).agg(label=('label', 'first'), vote=('pred', 'mean'), prob=('prob', 'mean'))
        user['pred'] = (user['vote'] >= 0.5).astype(int)
        y = user['label']
        p = user['pred']
        scores = {'accuracy': accuracy_score(y, p), 'precision_ASD': precision_score(y, p, zero_division=0), 'recall_ASD': recall_score(y, p, zero_division=0), 'f1_ASD': f1_score(y, p, zero_division=0), 'macro_f1': f1_score(y, p, average='macro', labels=[0, 1], zero_division=0), 'recall_non_ASD': recall_score(y, p, pos_label=0, zero_division=0), 'f1_non_ASD': f1_score(y, p, pos_label=0, zero_division=0), 'roc_auc': roc_auc_score(y, user['prob'])}
        return (scores, user)
    for seed in SEEDS:
        folder = ROOT / f'seed_{seed}'
        parts = {part: pd.read_csv(folder / f'{part}_users.csv') for part in ['train', 'val', 'test']}
        names = {part: set(table['username']) for part, table in parts.items()}
        assert not (names['train'] & names['val'] or names['train'] & names['test'] or names['val'] & names['test'])
        assert set().union(*names.values()) == set(dl_data['username'])
        train_full = dl_data[dl_data['username'].isin(names['train'])].copy()
        val_frame = dl_data[dl_data['username'].isin(names['val'])].reset_index(drop=True)
        test_frame = dl_data[dl_data['username'].isin(names['test'])].reset_index(drop=True)
        assert test_frame['username'].nunique() == 61
        assert test_frame.groupby('username')['label'].first().eq(0).sum() == 7
        if seed == 42:
            train_balanced = pd.read_pickle(BASE / '04_balanced_data/train_df_balanced.pkl')[['username', 'label', 'cleaned_tweet']].copy()
            assert set(train_balanced['username']).issubset(names['train'])
        else:
            n_minority = train_full['label'].value_counts().min()
            pieces = []
            for label, group in train_full.groupby('label'):
                if len(group) > n_minority:
                    group = group.sample(n=n_minority, random_state=seed)
                pieces.append(group)
            train_balanced = pd.concat(pieces).sample(frac=1, random_state=seed).reset_index(drop=True)
        assert train_balanced['label'].nunique() == 2
        assert train_balanced['label'].value_counts().nunique() == 1
        assert train_balanced['cleaned_tweet'].notna().all()
        tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
        tokenizer.fit_on_texts(train_balanced['cleaned_tweet'])
        truncation_rates = {}

        def encode_frame(frame, subset):
            sequences = tokenizer.texts_to_sequences(frame['cleaned_tweet'])
            truncation_rates[subset] = float(np.mean([len(s) > MAX_LEN for s in sequences]))
            return pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')
        X_train = encode_frame(train_balanced, 'train')
        X_val = encode_frame(val_frame, 'val')
        X_test = encode_frame(test_frame, 'test')
        y_train = train_balanced['label'].to_numpy(dtype='float32')
        y_val = val_frame['label'].to_numpy(dtype='float32')
        balanced_hash = hashlib.sha256(pd.util.hash_pandas_object(train_balanced.reset_index(drop=True), index=True).values.tobytes()).hexdigest()
        split_hash = hashlib.sha256(json.dumps({part: sorted(values) for part, values in names.items()}, sort_keys=True).encode()).hexdigest()
        (folder / 'PRIVATE_DL_tokenizer.json').write_text(tokenizer.to_json())
        for name in ['Bi-LSTM', 'TextCNN']:
            safe = name.replace('-', '_')
            result_path = folder / f'DL_{safe}_metrics.csv'
            config_path = folder / f'DL_{safe}_config.json'
            config = {'protocol': 'DL_multiseed_v1', 'model': name, 'seed': seed, 'data_hash': data_hash, 'balanced_training_hash': balanced_hash, 'split_hash': split_hash, 'versions': versions, 'max_words': MAX_WORDS, 'sequence_length': MAX_LEN, 'embedding_dimension': EMBED_DIM, 'padding': 'post', 'truncating': 'post', 'optimizer': 'Adam', 'learning_rate': 0.001, 'loss': 'binary_crossentropy', 'max_epochs': MAX_EPOCHS, 'batch_size': BATCH_SIZE, 'early_stopping_patience': 3, 'checkpoint_selection': 'minimum validation loss', 'tweet_threshold': 0.5, 'user_prediction': 'majority vote; tie -> ASD', 'user_ROC_AUC': 'mean tweet probability', 'truncation_fractions': truncation_rates}
            signature = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()
            if result_path.exists() and config_path.exists():
                previous = json.loads(config_path.read_text())
                assert previous['signature'] == signature, 'Artifact validation failed.'
                continue
            tf.keras.backend.clear_session()
            tf.keras.utils.set_random_seed(seed)
            model = build_dl_model(name)
            history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, shuffle=True, callbacks=[EarlyStopping(monitor='val_loss', mode='min', patience=3, restore_best_weights=True, verbose=1), CSVLogger(str(folder / f'DL_{safe}_history.csv'), append=False), TerminateOnNaN()], verbose=1)
            assert np.isfinite(history.history['loss']).all(), 'Artifact validation failed.'
            assert np.isfinite(history.history['val_loss']).all(), 'Artifact validation failed.'
            probability = model.predict(X_test, batch_size=256, verbose=1).reshape(-1)
            scores, user = evaluate_dl_users(test_frame, probability)
            model.save(str(folder / f'DL_{safe}_best.keras'))
            user.to_csv(folder / f'PRIVATE_DL_{safe}_user_predictions.csv')
            config['signature'] = signature
            config['epochs_run'] = len(history.history['loss'])
            config['best_epoch'] = int(np.argmin(history.history['val_loss']) + 1)
            config['architecture'] = json.loads(model.to_json())
            config['optimizer_config'] = model.optimizer.get_config()
            config_path.write_text(json.dumps(config, indent=2, default=str))
            pd.DataFrame([{'model': name, 'seed': seed, **scores}]).to_csv(result_path, index=False)
            del model, history, probability, user
            tf.keras.backend.clear_session()
            gc.collect()
        del X_train, X_val, X_test, y_train, y_val
        del train_full, train_balanced, val_frame, test_frame
        del tokenizer
        gc.collect()
    paths = [ROOT / f'seed_{seed}' / f'DL_{safe}_metrics.csv' for seed in SEEDS for safe in ['Bi_LSTM', 'TextCNN']]
    assert all((path.is_file() for path in paths))
    dl_results = pd.concat([pd.read_csv(path) for path in paths], ignore_index=True)
    assert len(dl_results) == 10
    assert not dl_results.duplicated(['model', 'seed']).any()
    dl_results.to_csv(ROOT / 'DL_multiseed_results.csv', index=False)
    metric_columns = ['accuracy', 'precision_ASD', 'recall_ASD', 'f1_ASD', 'macro_f1', 'recall_non_ASD', 'f1_non_ASD', 'roc_auc']
    dl_summary = dl_results.groupby('model')[metric_columns].agg(['mean', 'std'])
    dl_summary.to_csv(ROOT / 'DL_multiseed_mean_SD.csv')
    display(dl_results[['model', 'seed', 'macro_f1', 'roc_auc']])
    display(dl_summary[['macro_f1', 'roc_auc']].round(2))
    return dl_results


In [13]:
stage_functions = {
    'holdout_dl': train_holdout_dl,
    'standalone_dl': train_multiseed_dl
}
for selected, enabled in [(PREPARATION_STAGE,ENABLE_PREPROCESSING),(ANALYSIS_STAGE,ENABLE_ANALYSIS),(TRAINING_STAGE,ENABLE_TRAINING)]:
    if enabled is True and selected in stage_functions:
        stage_result = stage_functions[selected]()
        if stage_result is not None:
            display(stage_result)


## 5. Transformers

DistilBERT, RoBERTa, and BERTweet. Holdout and five-seed experiments use separate protocols.


In [14]:
def train_holdout_transformers():
    require_source_stage('holdout_transformer', 'training', outputs=['12_transformers/*'], inputs=['02_preprocessing/cleaned_data.pkl', '03_split/train_users.csv', '03_split/val_users.csv', '03_split/test_users.csv'])
    import os, re, html, glob, random, pickle
    import numpy as np
    import pandas as pd
    from pathlib import Path
    SAVE_DIR = str(BASE)
    SEED = 42
    df = pd.read_pickle(BASE / '02_preprocessing/cleaned_data.pkl')
    random.seed(SEED)
    np.random.seed(SEED)
    import gc, shutil
    import torch
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    DEVICE = torch.device('cuda')
    temporary_paths = [Path('/content/best_tmp')] + [Path(f'/content/tmp_{name}') for name in ['DistilBERT', 'RoBERTa', 'BERTweet']]
    if any((path.exists() for path in temporary_paths)):
        raise FileExistsError('Holdout Transformer temporary paths must be absent before this stage.')

    def _remove_owned_directory(path, ignore_errors=False):
        target = Path(path).resolve()
        stage_root = (BASE / '12_transformers').resolve()
        if target not in [p.resolve() for p in temporary_paths] and (not target.is_relative_to(stage_root)):
            raise RuntimeError('Directory is outside this stage.')
        shutil.rmtree(target, ignore_errors=ignore_errors)
    from sklearn.model_selection import train_test_split
    SPLIT_DIR = os.path.join(SAVE_DIR, '03_split')
    os.makedirs(SPLIT_DIR, exist_ok=True)
    TRAIN_USERS_PATH = os.path.join(SPLIT_DIR, 'train_users.csv')
    VAL_USERS_PATH = os.path.join(SPLIT_DIR, 'val_users.csv')
    TEST_USERS_PATH = os.path.join(SPLIT_DIR, 'test_users.csv')
    if os.path.exists(TRAIN_USERS_PATH) and os.path.exists(VAL_USERS_PATH) and os.path.exists(TEST_USERS_PATH):
        train_users = pd.read_csv(TRAIN_USERS_PATH)
        val_users = pd.read_csv(VAL_USERS_PATH)
        test_users = pd.read_csv(TEST_USERS_PATH)
    else:
        user_table = df.groupby('username', as_index=False)['label'].first()
        train_pool_users, test_users = train_test_split(user_table, test_size=0.25, random_state=SEED, stratify=user_table['label'])
        train_users, val_users = train_test_split(train_pool_users, test_size=0.2, random_state=SEED, stratify=train_pool_users['label'])
        train_users.to_csv(TRAIN_USERS_PATH, index=False)
        val_users.to_csv(VAL_USERS_PATH, index=False)
        test_users.to_csv(TEST_USERS_PATH, index=False)
    train_df_full = df[df['username'].isin(train_users['username'])].copy()
    val_df = df[df['username'].isin(val_users['username'])].copy()
    test_df = df[df['username'].isin(test_users['username'])].copy()
    for name, part in [('Train', train_df_full), ('Val', val_df), ('Test', test_df)]:
        pass
    train_set = set(train_df_full['username'])
    val_set = set(val_df['username'])
    test_set = set(test_df['username'])
    assert train_set.isdisjoint(val_set)
    assert train_set.isdisjoint(test_set)
    assert val_set.isdisjoint(test_set)
    all_users = set(df['username'])
    split_users = train_set | val_set | test_set
    assert all_users == split_users
    BALANCE_DIR = os.path.join(SAVE_DIR, '04_balanced_data')
    os.makedirs(BALANCE_DIR, exist_ok=True)
    TRAIN_BALANCED_PATH = os.path.join(BALANCE_DIR, 'train_df_balanced.pkl')
    if os.path.exists(TRAIN_BALANCED_PATH):
        train_df = pd.read_pickle(TRAIN_BALANCED_PATH)
    else:
        class_counts = train_df_full['label'].value_counts()
        minority_n = class_counts.min()
        train_parts = []
        for label_value, group in train_df_full.groupby('label'):
            if len(group) > minority_n:
                group = group.sample(n=minority_n, random_state=SEED)
            train_parts.append(group)
        train_df = pd.concat(train_parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
        train_df.to_pickle(TRAIN_BALANCED_PATH)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    import numpy as np, pandas as pd
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
    all_results = []

    def metric_dict(y_true, y_pred, y_prob, level, model_name):
        return {'model': model_name, 'level': level, 'accuracy': accuracy_score(y_true, y_pred), 'macro_precision': precision_score(y_true, y_pred, average='macro', zero_division=0), 'macro_recall': recall_score(y_true, y_pred, average='macro', zero_division=0), 'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0), 'roc_auc': roc_auc_score(y_true, y_prob)}

    def evaluate_tweet_and_user(data_df, pred, prob, model_name, save=True):
        temp = data_df[['username', 'label', 'cleaned_tweet']].copy().reset_index(drop=True)
        temp['pred'] = np.asarray(pred)
        temp['prob'] = np.asarray(prob)
        tweet_result = metric_dict(temp['label'], temp['pred'], temp['prob'], 'tweet', model_name)
        user_eval = temp.groupby('username').agg(label=('label', 'first'), vote_mean=('pred', 'mean'), prob=('prob', 'mean')).reset_index()
        user_eval['pred'] = (user_eval['vote_mean'] >= 0.5).astype(int)
        user_result = metric_dict(user_eval['label'], user_eval['pred'], user_eval['prob'], 'user', model_name)
        if save:
            all_results.extend([tweet_result, user_result])
        return (tweet_result, user_result, temp, user_eval)
    import shutil, time, pickle, psutil
    import numpy as np, torch
    from torch.utils.data import Dataset, DataLoader
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup, DataCollatorWithPadding
    from torch.optim import AdamW
    MAX_LEN_TRANSFORMER = 128
    BATCH_SIZE_TRANSFORMER = 32
    EPOCHS_TRANSFORMER = 2
    LR_TRANSFORMER = 2e-05
    TRANSFORMER_DIR = f'{SAVE_DIR}/12_transformers'
    os.makedirs(TRANSFORMER_DIR, exist_ok=True)
    transformer_objects, transformer_val_results, transformer_val_predictions = ({}, [], {})

    def mem_log(tag=''):
        ram = psutil.virtual_memory()
        gpu = torch.cuda.memory_allocated() / 1000000000.0

    def has_weights(d):
        return any((os.path.exists(os.path.join(d, f)) and os.path.getsize(os.path.join(d, f)) > 1000000.0 for f in ['model.safetensors', 'pytorch_model.bin']))

    class TweetDataset(Dataset):

        def __init__(self, texts, labels, tokenizer, max_len=128):
            self.enc = tokenizer(list(texts), truncation=True, max_length=max_len)
            self.labels = np.asarray(labels)

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, i):
            item = {k: v[i] for k, v in self.enc.items()}
            item['labels'] = int(self.labels[i])
            return item

    def train_transformer(model_name, train_df, val_df, epochs=EPOCHS_TRANSFORMER):
        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(DEVICE)
        collator = DataCollatorWithPadding(tokenizer)
        train_ds = TweetDataset(train_df['cleaned_tweet'], train_df['label'], tokenizer, MAX_LEN_TRANSFORMER)
        val_ds = TweetDataset(val_df['cleaned_tweet'], val_df['label'], tokenizer, MAX_LEN_TRANSFORMER)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE_TRANSFORMER, shuffle=True, collate_fn=collator)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE_TRANSFORMER * 2, shuffle=False, collate_fn=collator)
        optimizer = AdamW(model.parameters(), lr=LR_TRANSFORMER)
        total_steps = len(train_loader) * epochs
        scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * 0.1), total_steps)
        scaler = torch.amp.GradScaler('cuda')
        best_dir = '/content/best_tmp'
        _remove_owned_directory(best_dir, ignore_errors=True)
        best_val_loss = np.inf
        mem_log('Training')
        for epoch in range(epochs):
            model.train()
            train_loss = 0.0
            for step, batch in enumerate(train_loader):
                optimizer.zero_grad()
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                with torch.autocast('cuda', dtype=torch.float16):
                    loss = model(**batch).loss
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                train_loss += loss.item()
                if step % 200 == 0:
                    mem_log('Training')
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for batch in val_loader:
                    batch = {k: v.to(DEVICE) for k, v in batch.items()}
                    with torch.autocast('cuda', dtype=torch.float16):
                        val_loss += model(**batch).loss.item()
            avg_train, avg_val = (train_loss / len(train_loader), val_loss / len(val_loader))
            mem_log('Training')
            if avg_val < best_val_loss:
                best_val_loss = avg_val
                model.save_pretrained(best_dir)
        del model, optimizer
        gc.collect()
        torch.cuda.empty_cache()
        model = AutoModelForSequenceClassification.from_pretrained(best_dir).to(DEVICE)
        return (tokenizer, model)

    def transformer_predict(tokenizer, model, data_df):
        ds = TweetDataset(data_df['cleaned_tweet'], data_df['label'], tokenizer, MAX_LEN_TRANSFORMER)
        loader = DataLoader(ds, batch_size=BATCH_SIZE_TRANSFORMER * 2, shuffle=False, collate_fn=DataCollatorWithPadding(tokenizer))
        probs = []
        model.eval()
        with torch.no_grad():
            for batch in loader:
                batch = {k: v.to(DEVICE) for k, v in batch.items() if k != 'labels'}
                with torch.autocast('cuda', dtype=torch.float16):
                    logits = model(**batch).logits
                probs.extend(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
        probs = np.asarray(probs)
        return ((probs >= 0.5).astype(int), probs)

    def run_one_transformer(short_name, hf_name):
        model_dir = f'{TRANSFORMER_DIR}/{short_name}'
        tokenizer_dir, model_save_dir = (f'{model_dir}/tokenizer', f'{model_dir}/model')
        VAL_PROB_PATH, VAL_PRED_PATH, RESULT_PATH = (f'{model_dir}/val_prob.npy', f'{model_dir}/val_pred.npy', f'{model_dir}/validation_result.pkl')
        os.makedirs(model_dir, exist_ok=True)
        checkpoint_exists = os.path.isdir(tokenizer_dir) and has_weights(model_save_dir) and all((os.path.exists(p) for p in [VAL_PROB_PATH, VAL_PRED_PATH, RESULT_PATH]))
        if checkpoint_exists:
            tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, use_fast=True)
            model = AutoModelForSequenceClassification.from_pretrained(model_save_dir).to(DEVICE)
            val_prob, val_pred = (np.load(VAL_PROB_PATH), np.load(VAL_PRED_PATH))
            with open(RESULT_PATH, 'rb') as f:
                result = pickle.load(f)
        else:
            _remove_owned_directory(model_dir, ignore_errors=True)
            os.makedirs(model_dir, exist_ok=True)
            tokenizer, model = train_transformer(hf_name, train_df, val_df)
            tmp = f'/content/tmp_{short_name}'
            _remove_owned_directory(tmp, ignore_errors=True)
            model.save_pretrained(f'{tmp}/model')
            tokenizer.save_pretrained(f'{tmp}/tokenizer')
            shutil.copytree(f'{tmp}/model', model_save_dir, dirs_exist_ok=True)
            shutil.copytree(f'{tmp}/tokenizer', tokenizer_dir, dirs_exist_ok=True)
            time.sleep(5)
            assert has_weights(model_save_dir), 'Artifact validation failed.'
            mem_log('Training')
            val_pred, val_prob = transformer_predict(tokenizer, model, val_df)
            np.save(VAL_PROB_PATH, val_prob)
            np.save(VAL_PRED_PATH, val_pred)
            tweet_result, user_result, _, _ = evaluate_tweet_and_user(val_df, val_pred, val_prob, short_name, save=False)
            result = {**tweet_result, 'user_macro_f1': user_result['macro_f1'], 'user_accuracy': user_result['accuracy'], 'user_roc_auc': user_result['roc_auc']}
            with open(RESULT_PATH, 'wb') as f:
                pickle.dump(result, f)
        transformer_objects[short_name] = {'tokenizer': tokenizer, 'model': model}
        transformer_val_predictions[short_name] = {'pred': val_pred, 'prob': val_prob}
        transformer_val_results.append(result)
        return result
    for short, hf_id in [('RoBERTa', 'roberta-base'), ('BERTweet', 'vinai/bertweet-base'), ('DistilBERT', 'distilbert-base-uncased')]:
        run_one_transformer(short, hf_id)
    return pd.DataFrame(transformer_val_results)


In [15]:
def predict_holdout_transformers():
    require_source_stage('holdout_transformer_predictions', 'analysis', outputs=['14_stacking/test/*_test_prob.npy', '14_stacking/test/*_test_pred.npy'], inputs=['02_preprocessing/cleaned_data.pkl', '03_split/test_users.csv'])
    import os, re, html, glob, random, pickle
    import numpy as np
    import pandas as pd
    from pathlib import Path
    SAVE_DIR = str(BASE)
    SEED = 42
    import gc, torch
    DEVICE = torch.device('cuda')
    cols = ['username', 'label', 'cleaned_tweet']
    import shutil, time, pickle, psutil
    import numpy as np, torch
    from torch.utils.data import Dataset, DataLoader
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup, DataCollatorWithPadding
    from torch.optim import AdamW
    MAX_LEN_TRANSFORMER = 128
    BATCH_SIZE_TRANSFORMER = 32

    class TweetDataset(Dataset):

        def __init__(self, texts, labels, tokenizer, max_len=128):
            self.enc = tokenizer(list(texts), truncation=True, max_length=max_len)
            self.labels = np.asarray(labels)

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, i):
            item = {k: v[i] for k, v in self.enc.items()}
            item['labels'] = int(self.labels[i])
            return item

    def transformer_predict(tokenizer, model, data_df):
        ds = TweetDataset(data_df['cleaned_tweet'], data_df['label'], tokenizer, MAX_LEN_TRANSFORMER)
        loader = DataLoader(ds, batch_size=BATCH_SIZE_TRANSFORMER * 2, shuffle=False, collate_fn=DataCollatorWithPadding(tokenizer))
        probs = []
        model.eval()
        with torch.no_grad():
            for batch in loader:
                batch = {k: v.to(DEVICE) for k, v in batch.items() if k != 'labels'}
                with torch.autocast('cuda', dtype=torch.float16):
                    logits = model(**batch).logits
                probs.extend(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
        probs = np.asarray(probs)
        return ((probs >= 0.5).astype(int), probs)
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    test_users = set(pd.read_csv(f'{SAVE_DIR}/03_split/test_users.csv')['username'])
    df = pd.read_pickle(f'{SAVE_DIR}/02_preprocessing/cleaned_data.pkl')
    test_df = df.loc[df['username'].isin(test_users), cols].reset_index(drop=True)
    del df
    gc.collect()
    TEST_DIR = f'{SAVE_DIR}/14_stacking/test'
    os.makedirs(TEST_DIR, exist_ok=True)
    for n in ['DistilBERT', 'RoBERTa', 'BERTweet']:
        if os.path.exists(f'{TEST_DIR}/{n}_test_prob.npy'):
            continue
        d = f'{SAVE_DIR}/12_transformers/{n}'
        tok = AutoTokenizer.from_pretrained(f'{d}/tokenizer')
        mdl = AutoModelForSequenceClassification.from_pretrained(f'{d}/model').to(DEVICE)
        pred, prob = transformer_predict(tok, mdl, test_df)
        np.save(f'{TEST_DIR}/{n}_test_pred.npy', pred)
        np.save(f'{TEST_DIR}/{n}_test_prob.npy', prob)
        del mdl, tok
        gc.collect()
        torch.cuda.empty_cache()
    return pd.DataFrame([{'model': n, 'test_probabilities': str(Path(TEST_DIR) / f'{n}_test_prob.npy')} for n in ['DistilBERT', 'RoBERTa', 'BERTweet']])


In [16]:
stage_functions = {
    'holdout_transformer': train_holdout_transformers,
    'holdout_transformer_predictions': predict_holdout_transformers
}
for selected, enabled in [(PREPARATION_STAGE,ENABLE_PREPROCESSING),(ANALYSIS_STAGE,ENABLE_ANALYSIS),(TRAINING_STAGE,ENABLE_TRAINING)]:
    if enabled is True and selected in stage_functions:
        stage_result = stage_functions[selected]()
        if stage_result is not None:
            display(stage_result)


### Five-Seed Transformer Training

DistilBERT, RoBERTa, and BERTweet across five seeds. Maximum length 128, two epochs, AdamW 2e-5. Select the checkpoint using validation loss.


In [17]:
def train_transformers():
    require_training('transformer', meta=False)
    import os
    os.environ["USE_TF"] = "0"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

    from pathlib import Path
    import gc
    import json
    import random
    import hashlib
    import tempfile
    import platform
    import importlib.metadata as metadata

    import numpy as np
    import pandas as pd
    import torch

    from torch.utils.data import Dataset, DataLoader
    from transformers import (
        AutoConfig,
        AutoTokenizer,
        AutoModelForSequenceClassification,
        DataCollatorWithPadding,
        get_linear_schedule_with_warmup
    )
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score,
        f1_score, roc_auc_score
    )
    from tqdm.auto import tqdm
    from IPython.display import display


    assert torch.cuda.is_available(), (
        'GPU required.'
    )
    DEVICE = torch.device("cuda")
    print('GPU:', torch.cuda.get_device_name(0), flush=True)

    BASE = Path("/content/drive_research/MyDrive/thesis/saved_results")

    try:
        ready = (BASE / "02_preprocessing/cleaned_data.pkl").is_file()
    except OSError:
        ready = False

    if not ready:
        from google.colab import drive
        drive.mount("/content/drive_research")

    assert (BASE / "02_preprocessing/cleaned_data.pkl").is_file()

    ROOT = BASE / "17_conference_revision/multiseed"
    assert ROOT.is_dir(), 'Missing multiseed split directory.'


    data = pd.read_pickle(
        BASE / "02_preprocessing/cleaned_data.pkl"
    )[["username", "label", "cleaned_tweet"]].reset_index(drop=True)

    assert data["cleaned_tweet"].notna().all()
    assert data["label"].isin([0, 1]).all()
    assert data.groupby("username")["label"].nunique().eq(1).all()
    assert data["username"].nunique() == 244

    SEEDS = [42, 43, 44, 45, 46]

    HF_MODELS = {
        "DistilBERT": "distilbert-base-uncased",
        "RoBERTa": "roberta-base",
        "BERTweet": "vinai/bertweet-base"
    }

    MAX_LEN = 128
    TRAIN_BATCH = 32
    EVAL_BATCH = 64
    EPOCHS = 2
    LEARNING_RATE = 2e-5
    WARMUP_FRACTION = 0.10

    versions = {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda": torch.version.cuda
    }
    for package in [
        "numpy", "pandas", "scikit-learn",
        "transformers", "sentencepiece", "emoji"
    ]:
        versions[package] = metadata.version(package)


    from datetime import datetime, timezone
    import uuid
    resume_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
    assert versions["transformers"] == "4.57.1", versions
    assert versions["sentencepiece"] == "0.2.2", versions
    assert versions["emoji"] == "2.16.0", versions
    (ROOT / f"Transformer_resume_environment_{resume_id}.json").write_text(
        json.dumps({"versions": versions, "GPU": torch.cuda.get_device_name(0),
                   "note": "Python 3.13 patch-version differences are recorded; other recorded package versions must match existing runs."}, indent=2)
    )

    data_hash = hashlib.sha256(
        pd.util.hash_pandas_object(
            data, index=True
        ).values.tobytes()
    ).hexdigest()


    revision_path = ROOT / "Transformer_model_revisions.json"

    assert revision_path.is_file(), (
        'Missing Transformer_model_revisions.json in the multiseed directory.'
    )
    revisions = json.loads(revision_path.read_text())



    def set_seed(seed):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


    class TweetDataset(Dataset):
        def __init__(self, frame, tokenizer):
            self.encodings = tokenizer(
                frame["cleaned_tweet"].tolist(),
                truncation=True,
                max_length=MAX_LEN
            )
            self.labels = frame["label"].to_numpy(dtype=int)

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, index):
            item = {
                key: values[index]
                for key, values in self.encodings.items()
            }
            item["labels"] = int(self.labels[index])
            return item


    def truncation_fraction(texts, tokenizer):
        truncated = 0
        total = 0

        for start in range(0, len(texts), 2048):
            batch = texts.iloc[start:start + 2048].tolist()

            encoded = tokenizer(
                batch,
                truncation=False,
                add_special_tokens=True,
                verbose=False
            )

            lengths = [
                len(tokens) for tokens in encoded["input_ids"]
            ]
            truncated += sum(length > MAX_LEN for length in lengths)
            total += len(lengths)

        return truncated / total


    def evaluate_users(frame, probability):
        probability = np.asarray(probability).reshape(-1)

        assert len(probability) == len(frame)
        assert np.isfinite(probability).all()
        assert ((probability >= 0) & (probability <= 1)).all()

        temp = frame[["username", "label"]].copy()
        temp["pred"] = (probability >= 0.5).astype(int)
        temp["prob"] = probability

        user = temp.groupby("username", sort=True).agg(
            label=("label", "first"),
            vote=("pred", "mean"),
            prob=("prob", "mean")
        )

        user["pred"] = (user["vote"] >= 0.5).astype(int)

        y = user["label"]
        p = user["pred"]

        scores = {
            "accuracy": accuracy_score(y, p),
            "precision_ASD": precision_score(y, p, zero_division=0),
            "recall_ASD": recall_score(y, p, zero_division=0),
            "f1_ASD": f1_score(y, p, zero_division=0),
            "macro_f1": f1_score(
                y, p, average="macro",
                labels=[0, 1], zero_division=0
            ),
            "recall_non_ASD": recall_score(
                y, p, pos_label=0, zero_division=0
            ),
            "f1_non_ASD": f1_score(
                y, p, pos_label=0, zero_division=0
            ),
            "roc_auc": roc_auc_score(y, user["prob"])
        }

        return scores, user



    for resume_phase in ["verify", "train"]:
        for seed in SEEDS:
            folder = ROOT / f"seed_{seed}"

            parts = {
                part: pd.read_csv(folder / f"{part}_users.csv")
                for part in ["train", "val", "test"]
            }
            names = {
                part: set(table["username"])
                for part, table in parts.items()
            }

            assert not (
                names["train"] & names["val"]
                or names["train"] & names["test"]
                or names["val"] & names["test"]
            )
            assert set().union(*names.values()) == set(data["username"])

            train_full = data[
                data["username"].isin(names["train"])
            ].copy()

            val_frame = data[
                data["username"].isin(names["val"])
            ].reset_index(drop=True)

            test_frame = data[
                data["username"].isin(names["test"])
            ].reset_index(drop=True)

            assert test_frame["username"].nunique() == 61
            assert (
                test_frame.groupby("username")["label"]
                .first().eq(0).sum()
            ) == 7


            if seed == 42:
                train_balanced = pd.read_pickle(
                    BASE / "04_balanced_data/train_df_balanced.pkl"
                )[["username", "label", "cleaned_tweet"]].copy()

                assert set(
                    train_balanced["username"]
                ).issubset(names["train"])
            else:
                n_minority = train_full["label"].value_counts().min()
                pieces = []

                for label, group in train_full.groupby("label"):
                    if len(group) > n_minority:
                        group = group.sample(
                            n=n_minority,
                            random_state=seed
                        )
                    pieces.append(group)

                train_balanced = (
                    pd.concat(pieces)
                    .sample(frac=1, random_state=seed)
                    .reset_index(drop=True)
                )

            assert train_balanced["label"].nunique() == 2
            assert train_balanced["label"].value_counts().nunique() == 1
            assert train_balanced["cleaned_tweet"].notna().all()

            balanced_hash = hashlib.sha256(
                pd.util.hash_pandas_object(
                    train_balanced.reset_index(drop=True),
                    index=True
                ).values.tobytes()
            ).hexdigest()

            split_hash = hashlib.sha256(
                json.dumps({
                    part: sorted(values)
                    for part, values in names.items()
                }, sort_keys=True).encode()
            ).hexdigest()

            for name, hf_id in HF_MODELS.items():
                revision = revisions[name]["revision"]
                assert revisions[name]["hf_id"] == hf_id

                result_path = folder / f"TFM_{name}_metrics.csv"
                config_path = folder / f"TFM_{name}_config.json"

                config = {
                    "protocol": "Transformer_multiseed_v1",
                    "model": name,
                    "hf_id": hf_id,
                    "pretrained_revision": revision,
                    "seed": seed,
                    "data_hash": data_hash,
                    "balanced_training_hash": balanced_hash,
                    "split_hash": split_hash,
                    "versions": versions,
                    "max_length": MAX_LEN,
                    "train_batch_size": TRAIN_BATCH,
                    "eval_batch_size": EVAL_BATCH,
                    "epochs": EPOCHS,
                    "optimizer": "torch.optim.AdamW",
                    "learning_rate": LEARNING_RATE,
                    "weight_decay": 0.01,
                    "warmup_fraction": WARMUP_FRACTION,
                    "mixed_precision": "float16",
                    "checkpoint_selection": "minimum mean batch validation loss",
                    "tweet_threshold": 0.5,
                    "user_prediction": "majority vote; tie -> ASD",
                    "user_ROC_AUC": "mean tweet probability"
                }

                signature = hashlib.sha256(
                    json.dumps(config, sort_keys=True).encode()
                ).hexdigest()

                prediction_path = folder / f"PRIVATE_TFM_{name}_user_predictions.csv"
                completion_files = [result_path, config_path, prediction_path]
                present = [p.is_file() for p in completion_files]
                assert not any(present) or all(present), (
                    f'Incomplete artifacts: {name} | seed={seed}: {dict(zip([p.name for p in completion_files], present))}. Incomplete experiment artifacts.'
                )
                if all(present):
                    previous = json.loads(config_path.read_text())

                    differences = {
                        key: {"existing": previous.get(key), "current": value}
                        for key, value in config.items()
                        if key != "versions" and previous.get(key) != value
                    }
                    assert not differences, (name, seed, differences)
                    old_versions = previous["versions"]
                    assert set(old_versions) == set(versions), 'Package version keys do not match.'
                    for key, value in versions.items():
                        if key == "python":
                            assert old_versions[key].split(".")[:2] == value.split(".")[:2], ('Python major/minor version mismatch', old_versions[key], value)
                        else:
                            assert old_versions[key] == value, (
                                f'Version {key} mismatch: recorded={old_versions[key]}, current={value}. Package versions must match the experiment configuration.'
                            )
                    old_config_for_hash = dict(config)
                    old_config_for_hash["versions"] = old_versions
                    old_signature = hashlib.sha256(
                        json.dumps(old_config_for_hash, sort_keys=True).encode()
                    ).hexdigest()
                    assert previous["signature"] == old_signature, 'Configuration signature mismatch.'
                    saved_users = pd.read_csv(prediction_path, index_col=0)
                    expected_labels = test_frame.groupby("username", sort=True)["label"].first()
                    assert saved_users.index.is_unique
                    assert set(saved_users.index) == set(expected_labels.index)
                    assert np.array_equal(
                        saved_users.loc[expected_labels.index, "label"].to_numpy(),
                        expected_labels.to_numpy()
                    )
                    assert saved_users["pred"].isin([0, 1]).all()
                    assert np.isfinite(saved_users["prob"]).all()
                    assert saved_users["prob"].between(0, 1).all()
                    assert saved_users["vote"].between(0, 1).all()
                    assert np.array_equal(
                        saved_users["pred"].to_numpy(),
                        (saved_users["vote"] >= 0.5).astype(int).to_numpy()
                    )
                    result = pd.read_csv(result_path)
                    assert len(result) == 1 and result.iloc[0]["model"] == name
                    assert int(result.iloc[0]["seed"]) == seed
                    y, p = saved_users["label"], saved_users["pred"]
                    checks = {
                        "accuracy": accuracy_score(y, p),
                        "precision_ASD": precision_score(y, p, zero_division=0),
                        "recall_ASD": recall_score(y, p, zero_division=0),
                        "f1_ASD": f1_score(y, p, zero_division=0),
                        "macro_f1": f1_score(y, p, average="macro", labels=[0, 1], zero_division=0),
                        "recall_non_ASD": recall_score(y, p, pos_label=0, zero_division=0),
                        "f1_non_ASD": f1_score(y, p, pos_label=0, zero_division=0),
                        "roc_auc": roc_auc_score(y, saved_users["prob"])
                    }
                    for metric, value in checks.items():
                        assert np.isclose(float(result.iloc[0][metric]), value, atol=1e-8, rtol=0), (name, seed, metric, 'Metrics do not match saved predictions')
                    continue
                if resume_phase == "verify":
                    continue

                set_seed(seed)
                gc.collect()
                torch.cuda.empty_cache()

                print(f'{name} | seed={seed}', flush=True)

                tokenizer = AutoTokenizer.from_pretrained(
                    hf_id,
                    revision=revision,
                    use_fast=True
                )

                model = AutoModelForSequenceClassification.from_pretrained(
                    hf_id,
                    revision=revision,
                    num_labels=2
                ).to(DEVICE)

                frames = {
                    "train": train_balanced,
                    "val": val_frame,
                    "test": test_frame
                }


                truncation_rates = {
                    part: truncation_fraction(
                        frame["cleaned_tweet"], tokenizer
                    )
                    for part, frame in frames.items()
                }

                datasets = {
                    part: TweetDataset(frame, tokenizer)
                    for part, frame in frames.items()
                }

                collator = DataCollatorWithPadding(
                    tokenizer=tokenizer,
                    return_tensors="pt"
                )

                generator = torch.Generator().manual_seed(seed)

                loaders = {
                    "train": DataLoader(
                        datasets["train"],
                        batch_size=TRAIN_BATCH,
                        shuffle=True,
                        generator=generator,
                        collate_fn=collator,
                        num_workers=0
                    ),
                    "val": DataLoader(
                        datasets["val"],
                        batch_size=EVAL_BATCH,
                        shuffle=False,
                        collate_fn=collator,
                        num_workers=0
                    ),
                    "test": DataLoader(
                        datasets["test"],
                        batch_size=EVAL_BATCH,
                        shuffle=False,
                        collate_fn=collator,
                        num_workers=0
                    )
                }

                optimizer = torch.optim.AdamW(
                    model.parameters(),
                    lr=LEARNING_RATE,
                    weight_decay=0.01
                )

                total_steps = len(loaders["train"]) * EPOCHS
                warmup_steps = int(total_steps * WARMUP_FRACTION)

                scheduler = get_linear_schedule_with_warmup(
                    optimizer,
                    num_warmup_steps=warmup_steps,
                    num_training_steps=total_steps
                )

                scaler = torch.amp.GradScaler("cuda")
                history = []
                best_loss = float("inf")
                best_epoch = None


                with tempfile.TemporaryDirectory(
                    prefix="icimcis_transformer_"
                ) as temp_dir:

                    checkpoint = Path(temp_dir) / "best_state.pt"

                    for epoch in range(EPOCHS):
                        model.train()
                        training_loss = 0.0

                        for batch in tqdm(
                            loaders["train"],
                            desc=f"{name} seed {seed} epoch {epoch + 1}"
                        ):
                            batch = {
                                key: value.to(DEVICE)
                                for key, value in batch.items()
                            }

                            optimizer.zero_grad(set_to_none=True)

                            with torch.autocast(
                                device_type="cuda",
                                dtype=torch.float16
                            ):
                                loss = model(**batch).loss

                            assert torch.isfinite(loss).item(), (
                                'Non-finite training loss.'
                            )

                            scaler.scale(loss).backward()
                            scaler.step(optimizer)
                            scaler.update()
                            scheduler.step()

                            training_loss += float(loss.detach())

                        model.eval()
                        validation_loss = 0.0

                        with torch.no_grad():
                            for batch in tqdm(
                                loaders["val"],
                                desc=f'Validation {name} | seed={seed}'
                            ):
                                batch = {
                                    key: value.to(DEVICE)
                                    for key, value in batch.items()
                                }

                                with torch.autocast(
                                    device_type="cuda",
                                    dtype=torch.float16
                                ):
                                    val_loss = model(**batch).loss

                                assert torch.isfinite(val_loss).item()
                                validation_loss += float(val_loss)


                        avg_train = training_loss / len(loaders["train"])
                        avg_val = validation_loss / len(loaders["val"])

                        history.append({
                            "epoch": epoch + 1,
                            "train_loss": avg_train,
                            "val_loss": avg_val
                        })

                        pd.DataFrame(history).to_csv(
                            folder / f"TFM_{name}_history.csv",
                            index=False
                        )

                        print(f'{name} | seed={seed}, epoch {epoch + 1}: train_loss={avg_train:.4f}, val_loss={avg_val:.4f}', flush=True)

                        if avg_val < best_loss:
                            best_loss = avg_val
                            best_epoch = epoch + 1
                            torch.save(model.state_dict(), checkpoint)

                    assert checkpoint.is_file()


                    del optimizer, scheduler, scaler
                    gc.collect()
                    torch.cuda.empty_cache()

                    state = torch.load(
                        checkpoint,
                        map_location="cpu",
                        weights_only=True
                    )
                    model.load_state_dict(state)
                    del state

                    model.eval()
                    probabilities = []

                    with torch.no_grad():
                        for batch in tqdm(
                            loaders["test"],
                            desc=f'Test {name} | seed={seed}'
                        ):
                            batch = {
                                key: value.to(DEVICE)
                                for key, value in batch.items()
                                if key != "labels"
                            }

                            with torch.autocast(
                                device_type="cuda",
                                dtype=torch.float16
                            ):
                                logits = model(**batch).logits

                            prob = torch.softmax(
                                logits.float(), dim=1
                            )[:, 1]

                            probabilities.append(
                                prob.cpu().numpy()
                            )

                    probabilities = np.concatenate(probabilities)

                    scores, user = evaluate_users(
                        test_frame, probabilities
                    )

                    user.to_csv(
                        folder /
                        f"PRIVATE_TFM_{name}_user_predictions.csv"
                    )

                    config["signature"] = signature
                    config["best_epoch"] = best_epoch
                    config["best_validation_loss"] = best_loss
                    config["truncation_fractions"] = truncation_rates
                    config["model_config"] = model.config.to_dict()
                    config["tokenizer_class"] = type(tokenizer).__name__
                    config["tokenizer_normalization"] = getattr(
                        tokenizer, "normalization", None
                    )

                    config_path.write_text(
                        json.dumps(config, indent=2, default=str)
                    )


                    pd.DataFrame([{
                        "model": name,
                        "seed": seed,
                        **scores
                    }]).to_csv(result_path, index=False)

                    print(f"{name} | seed={seed} | Macro-F1={scores['macro_f1']:.3f} | ROC-AUC={scores['roc_auc']:.3f}", flush=True)

                del model, tokenizer, datasets, loaders, collator
                del probabilities, user
                gc.collect()
                torch.cuda.empty_cache()

            del train_full, train_balanced, val_frame, test_frame
            gc.collect()



    paths = [
        ROOT / f"seed_{seed}" / f"TFM_{name}_metrics.csv"
        for seed in SEEDS
        for name in HF_MODELS
    ]

    assert all(path.is_file() for path in paths)

    transformer_results = pd.concat(
        [pd.read_csv(path) for path in paths],
        ignore_index=True
    )

    assert len(transformer_results) == 15
    assert not transformer_results.duplicated(
        ["model", "seed"]
    ).any()

    transformer_results.to_csv(
        ROOT / "Transformer_multiseed_results.csv",
        index=False
    )

    metric_columns = [
        "accuracy", "precision_ASD", "recall_ASD", "f1_ASD",
        "macro_f1", "recall_non_ASD", "f1_non_ASD", "roc_auc"
    ]

    summary = transformer_results.groupby(
        "model"
    )[metric_columns].agg(["mean", "std"])

    summary.to_csv(
        ROOT / "Transformer_multiseed_mean_SD.csv"
    )

    display(transformer_results[
        ["model", "seed", "macro_f1", "roc_auc"]
    ])
    display(summary[["macro_f1", "roc_auc"]].round(2))




    family_files = [
        ROOT / "ML_multiseed_results.csv",
        ROOT / "DL_multiseed_results.csv",
        ROOT / "Transformer_multiseed_results.csv"
    ]

    if all(path.is_file() for path in family_files):
        combined = pd.concat(
            [pd.read_csv(path) for path in family_files],
            ignore_index=True
        )

        assert len(combined) == 40
        assert not combined.duplicated(["model", "seed"]).any()
        assert combined.groupby("model")["seed"].nunique().eq(5).all()

        combined.to_csv(
            ROOT / "All_models_multiseed_results.csv",
            index=False
        )

        combined_summary = combined.groupby(
            "model"
        )[metric_columns].agg(["mean", "std"])

        combined_summary.to_csv(
            ROOT / "All_models_multiseed_mean_SD.csv"
        )

        display(
            combined_summary[
                ["macro_f1", "roc_auc"]
            ].round(2)
        )



In [18]:
if ENABLE_TRAINING and TRAINING_STAGE == 'transformer':
    train_transformers()


## 6. Prediction and Evaluation

Tweet- and user-level evaluation across five seeds.


In [19]:
def scores(y, prediction, probability):
    return {
        'accuracy': accuracy_score(y, prediction),
        'precision_ASD': precision_score(y, prediction, zero_division=0),
        'recall_ASD': recall_score(y, prediction, zero_division=0),
        'f1_ASD': f1_score(y, prediction, zero_division=0),
        'macro_f1': f1_score(y, prediction, average='macro', labels=[0, 1], zero_division=0),
        'recall_non_ASD': recall_score(y, prediction, pos_label=0, zero_division=0),
        'f1_non_ASD': f1_score(y, prediction, pos_label=0, zero_division=0),
        'roc_auc': roc_auc_score(y, probability)
    }


In [20]:
HF_MODELS = {'DistilBERT':'distilbert-base-uncased','RoBERTa':'roberta-base','BERTweet':'vinai/bertweet-base'}

def load_transformer_users(data,name,seed):
    folder = MULTISEED/f'seed_{seed}'
    require_files([folder/f'TFM_{name}_config.json',folder/f'TFM_{name}_metrics.csv',
                   folder/f'PRIVATE_TFM_{name}_user_predictions.csv'])
    config = read_json(folder/f'TFM_{name}_config.json')
    assert config['protocol']=='Transformer_multiseed_v1' and config['model']==name and config['seed']==seed
    assert config['hf_id']==HF_MODELS[name]
    revisions = read_json(MULTISEED/'Transformer_model_revisions.json')
    assert revisions[name]['hf_id']==config['hf_id'] and revisions[name]['revision']==config['pretrained_revision']
    parts,positions = partitions[seed]
    data_hash = hashlib.sha256(pd.util.hash_pandas_object(data,index=True).values.tobytes()).hexdigest()
    split_hash = hashlib.sha256(json.dumps({p:sorted(v) for p,v in parts.items()},sort_keys=True).encode()).hexdigest()
    assert config['data_hash']==data_hash and config['split_hash']==split_hash
    train_full = data[data['username'].isin(parts['train'])].copy()
    if seed==42:
        require_files([BASE/'04_balanced_data/train_df_balanced.pkl'])
        train_balanced = pd.read_pickle(BASE/'04_balanced_data/train_df_balanced.pkl')[['username','label','cleaned_tweet']].copy()
        assert set(train_balanced['username']).issubset(parts['train'])
    else:
        minimum = train_full['label'].value_counts().min()
        pieces = [group.sample(n=minimum,random_state=seed) if len(group)>minimum else group
                  for _,group in train_full.groupby('label')]
        train_balanced = pd.concat(pieces).sample(frac=1,random_state=seed).reset_index(drop=True)
    assert train_balanced['label'].value_counts().nunique()==1
    assert config['balanced_training_hash']==hashlib.sha256(pd.util.hash_pandas_object(
        train_balanced.reset_index(drop=True),index=True).values.tobytes()).hexdigest()
    signed_keys = ['protocol','model','hf_id','pretrained_revision','seed','data_hash','balanced_training_hash',
                   'split_hash','versions','max_length','train_batch_size','eval_batch_size','epochs','optimizer',
                   'learning_rate','weight_decay','warmup_fraction','mixed_precision','checkpoint_selection',
                   'tweet_threshold','user_prediction','user_ROC_AUC']
    signed = {key:config[key] for key in signed_keys}
    assert config['signature']==hashlib.sha256(json.dumps(signed,sort_keys=True).encode()).hexdigest()
    expected={'max_length':128,'train_batch_size':32,'eval_batch_size':64,'epochs':2,'learning_rate':2e-5,
              'weight_decay':0.01,'warmup_fraction':0.10,'mixed_precision':'float16',
              'checkpoint_selection':'minimum mean batch validation loss','tweet_threshold':0.5}
    assert all(config[key]==value for key,value in expected.items())
    users = pd.read_csv(folder/f'PRIVATE_TFM_{name}_user_predictions.csv',index_col=0)
    users = check_users(users,data,positions['test'],prediction='pred',probability='prob',vote='vote')
    row = pd.read_csv(folder/f'TFM_{name}_metrics.csv')
    assert len(row)==1 and row.iloc[0]['model']==name and int(row.iloc[0]['seed'])==seed
    values = scores(users['label'],users['pred'],users['prob'])
    check_metric_row(row.iloc[0],values)
    del train_full,train_balanced
    gc.collect()
    return values,users


In [21]:
rows = []
for seed in SEEDS:
    for name in HF_MODELS:
        values,users = load_transformer_users(data,name,seed)
        matrix = confusion_matrix(users['label'],users['pred'],labels=[0,1])
        rows.append({'model':name,'seed':seed,**values,'TN':int(matrix[0,0]),'FP':int(matrix[0,1]),'FN':int(matrix[1,0]),'TP':int(matrix[1,1])})
        del users
transformer_results = pd.DataFrame(rows)
summary = transformer_results.groupby('model')[METRICS].agg(['mean','std'])
display(transformer_results[['model','seed']+METRICS+['TN','FP','FN','TP']])
display(summary.round(4))


,model,seed,accuracy,precision_ASD,recall_ASD,f1_ASD,macro_f1,recall_non_ASD,f1_non_ASD,roc_auc,TN,FP,FN,TP
0,DistilBERT,42,0.819672,0.905660,0.888889,0.897196,0.581931,0.285714,0.266667,0.798942,2,5,6,48
1,RoBERTa,42,0.836066,0.892857,0.925926,0.909091,0.537879,0.142857,0.166667,0.777778,1,6,4,50
2,BERTweet,42,0.868852,0.910714,0.944444,0.927273,0.630303,0.285714,0.333333,0.746032,2,5,3,51
3,DistilBERT,43,0.770492,0.884615,0.851852,0.867925,0.496462,0.142857,0.125000,0.505291,1,6,8,46
4,RoBERTa,43,0.754098,0.882353,0.833333,0.857143,0.487395,0.142857,0.117647,0.529101,1,6,9,45
5,BERTweet,43,0.770492,0.900000,0.833333,0.865385,0.543803,0.285714,0.222222,0.568783,2,5,9,45
6,DistilBERT,44,0.868852,0.910714,0.944444,0.927273,0.630303,0.285714,0.333333,0.677249,2,5,3,51
7,RoBERTa,44,0.754098,0.914894,0.796296,0.851485,0.568600,0.428571,0.285714,0.698413,3,4,11,43
8,BERTweet,44,0.852459,0.941176,0.888889,0.914286,0.692437,0.571429,0.470588,0.671958,4,3,6,48
9,DistilBERT,45,0.934426,0.962963,0.962963,0.962963,0.838624,0.714286,0.714286,0.978836,5,2,2,52


accuracy         precision_ASD         recall_ASD          f1_ASD  \
               mean     std          mean     std       mean     std    mean   
model                                                                          
BERTweet     0.8557  0.0650        0.9281  0.0336     0.9074  0.0507  0.9173   
DistilBERT   0.8328  0.0699        0.9097  0.0321     0.9000  0.0517  0.9047   
RoBERTa      0.8197  0.0811        0.9085  0.0331     0.8852  0.0734  0.8958   

                   macro_f1         recall_non_ASD         f1_non_ASD         \
               std     mean     std           mean     std       mean    std   
model                                                                          
BERTweet    0.0384   0.6698  0.1323         0.4571  0.2556     0.4224  0.230   
DistilBERT  0.0408   0.6088  0.1408         0.3143  0.2347     0.3129  0.242   
RoBERTa     0.0489   0.5961  0.1564         0.3143  0.2556     0.2964  0.272   

           roc_auc          
              mean     std  
model                       
BERTweet    0.7508  0.1491  
DistilBERT  0.7201  0.1786  
RoBERTa     0.7196  0.1694

In [22]:
def holdout_full_metrics():
    table = read_report_csv(BASE/'HOLDOUT_FINAL/15_final_results/all_results.csv')
    table = table.loc[table['model'].isin(REPORT_BASE_MODELS)].copy()
    return table.rename(columns={'precision':'precision_ASD', 'recall':'recall_ASD',
                                 'f1':'f1_ASD'}).round(4)

def holdout_full_intervals():
    table = read_report_csv(REVISION/'user_level_bootstrap_CI.csv')
    return table.loc[table['model'].isin(REPORT_BASE_MODELS)].round(4)

def linguistic_baseline_metrics():
    table = read_report_csv(REVISION/'interpretable_baseline/linguistic_baseline_multiseed_results.csv')
    assert not table.duplicated(['model','seed']).any()
    display(table.round(4))
    return table.groupby('model')[METRICS].agg(['mean','std']).round(4)

def transformer_training_summary():
    rows = []
    for seed in SEEDS:
        for model in HF_MODELS:
            config = read_json(MULTISEED/f'seed_{seed}/TFM_{model}_config.json')
            row = {'model':model,'seed':seed,'best_epoch':config.get('best_epoch')}
            row.update({f'{split}_truncation_fraction':value
                        for split,value in config.get('truncation_fractions',{}).items()})
            rows.append(row)
    return pd.DataFrame(rows).round(6)


In [23]:
show_result('Transformer Checkpoints and Truncation',transformer_training_summary)


Transformer Checkpoints and Truncation


,model,seed,best_epoch,train_truncation_fraction,val_truncation_fraction,test_truncation_fraction
0,DistilBERT,42,2,0.000000,0.000000,0.000004
1,RoBERTa,42,2,0.000010,0.000016,0.000020
2,BERTweet,42,2,0.000000,0.000000,0.000004
3,DistilBERT,43,1,0.000000,0.000006,0.000009
4,RoBERTa,43,1,0.000018,0.000000,0.000009
5,BERTweet,43,2,0.000000,0.000006,0.000000
6,DistilBERT,44,1,0.000000,0.000000,0.000004
7,RoBERTa,44,1,0.000009,0.000009,0.000000
8,BERTweet,44,1,0.000000,0.000000,0.000004
9,DistilBERT,45,1,0.000010,0.000000,0.000000


,model,seed,best_epoch,train_truncation_fraction,val_truncation_fraction,test_truncation_fraction
0,DistilBERT,42,2,0.000000,0.000000,0.000004
1,RoBERTa,42,2,0.000010,0.000016,0.000020
2,BERTweet,42,2,0.000000,0.000000,0.000004
3,DistilBERT,43,1,0.000000,0.000006,0.000009
4,RoBERTa,43,1,0.000018,0.000000,0.000009
5,BERTweet,43,2,0.000000,0.000006,0.000000
6,DistilBERT,44,1,0.000000,0.000000,0.000004
7,RoBERTa,44,1,0.000009,0.000009,0.000000
8,BERTweet,44,1,0.000000,0.000000,0.000004
9,DistilBERT,45,1,0.000010,0.000000,0.000000


### Base Model Summary - Table IV

Mean and sample SD of the five-seed ML/DL results.


In [24]:
family_files = [MULTISEED/'ML_multiseed_results.csv',MULTISEED/'DL_multiseed_results.csv']
require_files(family_files)
combined = pd.concat([pd.read_csv(path) for path in family_files]+[transformer_results],ignore_index=True)
assert len(combined)==40 and not combined.duplicated(['model','seed']).any()
assert combined.groupby('model')['seed'].nunique().eq(5).all()
combined_summary = combined.groupby('model')[METRICS].agg(['mean','std'])
display(combined_summary.round(4))


accuracy         precision_ASD         recall_ASD          \
                        mean     std          mean     std       mean     std   
model                                                                           
BERTweet              0.8557  0.0650        0.9281  0.0336     0.9074  0.0507   
Bi-LSTM               0.8754  0.0340        0.9259  0.0471     0.9370  0.0248   
DistilBERT            0.8328  0.0699        0.9097  0.0321     0.9000  0.0517   
Logistic Regression   0.8492  0.0408        0.9121  0.0227     0.9185  0.0384   
Random Forest         0.8689  0.0201        0.8941  0.0117     0.9667  0.0275   
RoBERTa               0.8197  0.0811        0.9085  0.0331     0.8852  0.0734   
SVM                   0.8590  0.0411        0.9137  0.0327     0.9296  0.0356   
TextCNN               0.8557  0.0374        0.9102  0.0257     0.9296  0.0401   

                     f1_ASD         macro_f1         recall_non_ASD          \
                       mean     std     mean     std           mean     std   
model                                                                         
BERTweet             0.9173  0.0384   0.6698  0.1323         0.4571  0.2556   
Bi-LSTM              0.9305  0.0179   0.6438  0.1555         0.4000  0.3964   
DistilBERT           0.9047  0.0408   0.6088  0.1408         0.3143  0.2347   
Logistic Regression  0.9149  0.0239   0.6168  0.0917         0.3143  0.1863   
Random Forest        0.9287  0.0117   0.5401  0.0730         0.1143  0.1195   
RoBERTa              0.8958  0.0489   0.5961  0.1564         0.3143  0.2556   
SVM                  0.9211  0.0228   0.6196  0.1290         0.3143  0.2748   
TextCNN              0.9192  0.0217   0.6074  0.1077         0.2857  0.2259   

                    f1_non_ASD         roc_auc          
                          mean     std    mean     std  
model                                                   
BERTweet                0.4224  0.2300  0.7508  0.1491  
Bi-LSTM                 0.3571  0.2967  0.7926  0.1689  
DistilBERT              0.3129  0.2420  0.7201  0.1786  
Logistic Regression     0.3187  0.1652  0.7772  0.1707  
Random Forest           0.1515  0.1435  0.7905  0.1654  
RoBERTa                 0.2964  0.2720  0.7196  0.1694  
SVM                     0.3181  0.2387  0.7868  0.1729  
TextCNN                 0.2956  0.2012  0.7550  0.1607

## 7. Holdout Results - Table III

Tweet- and user-level results with user bootstrap CIs. This holdout experiment is distinct from the five-seed evaluation above. OOF stacking is evaluated in notebook 02.


In [25]:
def holdout_results():
    metrics = read_report_csv(BASE / 'HOLDOUT_FINAL/15_final_results/all_results.csv')
    intervals = read_report_csv(REVISION / 'user_level_bootstrap_CI.csv')
    rows = []
    for model in REPORT_BASE_MODELS:
        part = metrics.loc[metrics['model'].eq(model)].set_index('level')
        assert set(part.index) == {'tweet', 'user'} and len(part) == 2
        row = {'model': model,
               'tweet_macro_f1': part.loc['tweet', 'macro_f1'],
               'tweet_roc_auc': part.loc['tweet', 'roc_auc']}
        ci = intervals.loc[intervals['model'].eq(model)].set_index('metric')
        for metric in ['macro_f1', 'roc_auc']:
            np.testing.assert_allclose(part.loc['user', metric], ci.loc[metric, 'estimate'])
            row.update({f'user_{metric}': part.loc['user', metric],
                        f'{metric}_CI_low': ci.loc[metric, 'CI_lower'],
                        f'{metric}_CI_high': ci.loc[metric, 'CI_upper']})
        rows.append(row)
    return pd.DataFrame(rows).round(6)


In [26]:
show_result('Base Models - Table III',holdout_results)
show_result('Holdout Metrics',holdout_full_metrics)
show_result('User Bootstrap Intervals',holdout_full_intervals)


Base Models - Table III


,model,tweet_macro_f1,tweet_roc_auc,user_macro_f1,macro_f1_CI_low,macro_f1_CI_high,user_roc_auc,roc_auc_CI_low,roc_auc_CI_high
0,Logistic Regression,0.487081,0.621683,0.612562,0.445455,0.784757,0.888889,0.796296,0.960317
1,SVM,0.485474,0.622428,0.612562,0.445455,0.784757,0.910053,0.825397,0.973611
2,Random Forest,0.497879,0.598739,0.464912,0.455357,0.469565,0.883598,0.783069,0.960317
3,Bi-LSTM,0.471036,0.633610,0.739316,0.596561,0.885981,0.886243,0.775132,0.968254
4,TextCNN,0.476601,0.638259,0.657128,0.505919,0.804487,0.880952,0.777712,0.962963
5,DistilBERT,0.490306,0.624932,0.612562,0.445455,0.784757,0.785714,0.595238,0.931217
6,RoBERTa,0.494942,0.619841,0.537879,0.435185,0.698659,0.761905,0.552910,0.933862
7,BERTweet,0.486237,0.549099,0.469565,0.469565,0.469565,0.500000,0.283069,0.706415


Holdout Metrics


,model,level,accuracy,precision_ASD,recall_ASD,f1_ASD,macro_f1,roc_auc
0,Logistic Regression,tweet,0.6745,0.9381,0.6929,0.7971,0.4871,0.6217
1,Logistic Regression,user,0.8525,0.9091,0.9259,0.9174,0.6126,0.8889
2,SVM,tweet,0.6668,0.9394,0.6830,0.7910,0.4855,0.6224
3,SVM,user,0.8525,0.9091,0.9259,0.9174,0.6126,0.9101
4,Random Forest,tweet,0.7142,0.9342,0.7426,0.8275,0.4979,0.5987
5,Random Forest,user,0.8689,0.8833,0.9815,0.9298,0.4649,0.8836
6,Bi-LSTM,tweet,0.6269,0.9433,0.6338,0.7582,0.4710,0.6336
7,Bi-LSTM,user,0.8689,0.9600,0.8889,0.9231,0.7393,0.8862
8,TextCNN,tweet,0.6386,0.9431,0.6475,0.7678,0.4766,0.6383
9,TextCNN,user,0.8197,0.9388,0.8519,0.8932,0.6571,0.8810


User Bootstrap Intervals


,model,metric,estimate,CI_lower,CI_upper,n_users,n_non_ASD,bootstrap_repetitions
0,Logistic Regression,accuracy,0.8525,0.7705,0.9180,61,7,5000
1,Logistic Regression,precision_ASD,0.9091,0.8750,0.9464,61,7,5000
2,Logistic Regression,recall_ASD,0.9259,0.8519,0.9815,61,7,5000
3,Logistic Regression,f1_ASD,0.9174,0.8704,0.9550,61,7,5000
4,Logistic Regression,macro_f1,0.6126,0.4455,0.7848,61,7,5000
...,...,...,...,...,...,...,...,...
59,BERTweet,f1_ASD,0.9391,0.9391,0.9391,61,7,5000
60,BERTweet,macro_f1,0.4696,0.4696,0.4696,61,7,5000
61,BERTweet,recall_non_ASD,0.0000,0.0000,0.0000,61,7,5000
62,BERTweet,f1_non_ASD,0.0000,0.0000,0.0000,61,7,5000


,model,metric,estimate,CI_lower,CI_upper,n_users,n_non_ASD,bootstrap_repetitions
0,Logistic Regression,accuracy,0.8525,0.7705,0.9180,61,7,5000
1,Logistic Regression,precision_ASD,0.9091,0.8750,0.9464,61,7,5000
2,Logistic Regression,recall_ASD,0.9259,0.8519,0.9815,61,7,5000
3,Logistic Regression,f1_ASD,0.9174,0.8704,0.9550,61,7,5000
4,Logistic Regression,macro_f1,0.6126,0.4455,0.7848,61,7,5000
...,...,...,...,...,...,...,...,...
59,BERTweet,f1_ASD,0.9391,0.9391,0.9391,61,7,5000
60,BERTweet,macro_f1,0.4696,0.4696,0.4696,61,7,5000
61,BERTweet,recall_non_ASD,0.0000,0.0000,0.0000,61,7,5000
62,BERTweet,f1_non_ASD,0.0000,0.0000,0.0000,61,7,5000


### Holdout Test Prediction

Predict using model checkpoints and their matching text representations.


In [27]:
def evaluate_holdout_models():
    require_source_stage('holdout_evaluation', 'analysis', outputs=['15_final_results/*', 'HOLDOUT_FINAL/15_final_results/*'], inputs=['02_preprocessing/cleaned_data.pkl', '03_split/train_users.csv', '03_split/val_users.csv', '03_split/test_users.csv', '04_balanced_data/train_df_balanced.pkl', '06_tfidf/tfidf_vectorizer.pkl', '06_tfidf/X_train_tfidf.npz', '06_tfidf/X_val_tfidf.npz', '06_tfidf/X_test_tfidf.npz', '06_tfidf/y_train.npy', '06_tfidf/y_val.npy', '06_tfidf/y_test.npy', '07_ml_models/logistic_regression.pkl', '07_ml_models/svm_calibrated.pkl', '07_ml_models/random_forest.pkl', '09_tokenizer/keras_tokenizer.pkl', '09_tokenizer/X_train_seq.npy', '09_tokenizer/X_val_seq.npy', '09_tokenizer/X_test_seq.npy', '09_tokenizer/tokenizer_params.pkl', '10_dl_models/bilstm/best_model.keras', '10_dl_models/bilstm/history.pkl', '10_dl_models/bilstm/val_prob.npy', '10_dl_models/bilstm/val_pred.npy', '10_dl_models/textcnn/best_model.keras', '10_dl_models/textcnn/history.pkl', '10_dl_models/textcnn/val_prob.npy', '10_dl_models/textcnn/val_pred.npy', '14_stacking/test/DistilBERT_test_pred.npy', '14_stacking/test/DistilBERT_test_prob.npy', '14_stacking/test/RoBERTa_test_pred.npy', '14_stacking/test/RoBERTa_test_prob.npy', '14_stacking/test/BERTweet_test_pred.npy', '14_stacking/test/BERTweet_test_prob.npy'])
    import os, re, html, glob, random, pickle
    import numpy as np
    import pandas as pd
    from pathlib import Path
    SAVE_DIR = str(BASE)
    SEED = 42
    df = pd.read_pickle(BASE / '02_preprocessing/cleaned_data.pkl')
    random.seed(SEED)
    np.random.seed(SEED)

    def inference_fit_blocked():
        raise RuntimeError('Fitting is disabled in holdout_evaluation. Complete the cached artifacts first.')
    from sklearn.model_selection import train_test_split
    SPLIT_DIR = os.path.join(SAVE_DIR, '03_split')
    os.makedirs(SPLIT_DIR, exist_ok=True)
    TRAIN_USERS_PATH = os.path.join(SPLIT_DIR, 'train_users.csv')
    VAL_USERS_PATH = os.path.join(SPLIT_DIR, 'val_users.csv')
    TEST_USERS_PATH = os.path.join(SPLIT_DIR, 'test_users.csv')
    if os.path.exists(TRAIN_USERS_PATH) and os.path.exists(VAL_USERS_PATH) and os.path.exists(TEST_USERS_PATH):
        train_users = pd.read_csv(TRAIN_USERS_PATH)
        val_users = pd.read_csv(VAL_USERS_PATH)
        test_users = pd.read_csv(TEST_USERS_PATH)
    else:
        user_table = df.groupby('username', as_index=False)['label'].first()
        train_pool_users, test_users = train_test_split(user_table, test_size=0.25, random_state=SEED, stratify=user_table['label'])
        train_users, val_users = train_test_split(train_pool_users, test_size=0.2, random_state=SEED, stratify=train_pool_users['label'])
        train_users.to_csv(TRAIN_USERS_PATH, index=False)
        val_users.to_csv(VAL_USERS_PATH, index=False)
        test_users.to_csv(TEST_USERS_PATH, index=False)
    train_df_full = df[df['username'].isin(train_users['username'])].copy()
    val_df = df[df['username'].isin(val_users['username'])].copy()
    test_df = df[df['username'].isin(test_users['username'])].copy()
    for name, part in [('Train', train_df_full), ('Val', val_df), ('Test', test_df)]:
        pass
    train_set = set(train_df_full['username'])
    val_set = set(val_df['username'])
    test_set = set(test_df['username'])
    assert train_set.isdisjoint(val_set)
    assert train_set.isdisjoint(test_set)
    assert val_set.isdisjoint(test_set)
    all_users = set(df['username'])
    split_users = train_set | val_set | test_set
    assert all_users == split_users
    BALANCE_DIR = os.path.join(SAVE_DIR, '04_balanced_data')
    os.makedirs(BALANCE_DIR, exist_ok=True)
    TRAIN_BALANCED_PATH = os.path.join(BALANCE_DIR, 'train_df_balanced.pkl')
    if os.path.exists(TRAIN_BALANCED_PATH):
        train_df = pd.read_pickle(TRAIN_BALANCED_PATH)
    else:
        class_counts = train_df_full['label'].value_counts()
        minority_n = class_counts.min()
        train_parts = []
        for label_value, group in train_df_full.groupby('label'):
            if len(group) > minority_n:
                group = group.sample(n=minority_n, random_state=SEED)
            train_parts.append(group)
        train_df = pd.concat(train_parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
        train_df.to_pickle(TRAIN_BALANCED_PATH)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    import os
    import numpy as np
    import pandas as pd
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

    def metric_dict(y_true, y_pred, y_prob=None):
        result = {'accuracy': accuracy_score(y_true, y_pred), 'precision': precision_score(y_true, y_pred, zero_division=0), 'recall': recall_score(y_true, y_pred, zero_division=0), 'f1': f1_score(y_true, y_pred, zero_division=0), 'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0)}
        if y_prob is not None:
            try:
                result['roc_auc'] = roc_auc_score(y_true, y_prob)
            except:
                result['roc_auc'] = np.nan
        return result

    def evaluate_tweet_and_user(data_df, pred, prob, model_name, save=True):
        y_true = data_df['label'].values
        tweet_metrics = metric_dict(y_true, pred, prob)
        tweet_result = {'model': model_name, 'level': 'tweet', **tweet_metrics}
        user_eval = data_df[['username', 'label']].copy()
        user_eval['pred'] = pred
        user_eval['prob'] = prob
        user_eval = user_eval.groupby('username').agg(label=('label', 'first'), pred=('pred', 'mean'), prob=('prob', 'mean')).reset_index()
        user_eval['pred'] = (user_eval['pred'] >= 0.5).astype(int)
        user_metrics = metric_dict(user_eval['label'].values, user_eval['pred'].values, user_eval['prob'].values)
        user_result = {'model': model_name, 'level': 'user', **user_metrics}
        if save:
            if 'all_results' not in locals():
                locals()['all_results'] = []
            all_results.append(tweet_result)
            all_results.append(user_result)
        return (tweet_result, user_result, tweet_metrics, user_eval)
    from sklearn.feature_extraction.text import TfidfVectorizer
    import joblib
    TFIDF_DIR = os.path.join(SAVE_DIR, '06_tfidf')
    os.makedirs(TFIDF_DIR, exist_ok=True)
    TFIDF_MODEL_PATH = os.path.join(TFIDF_DIR, 'tfidf_vectorizer.pkl')
    X_TRAIN_TFIDF_PATH = os.path.join(TFIDF_DIR, 'X_train_tfidf.npz')
    X_VAL_TFIDF_PATH = os.path.join(TFIDF_DIR, 'X_val_tfidf.npz')
    X_TEST_TFIDF_PATH = os.path.join(TFIDF_DIR, 'X_test_tfidf.npz')
    Y_TRAIN_PATH = os.path.join(TFIDF_DIR, 'y_train.npy')
    Y_VAL_PATH = os.path.join(TFIDF_DIR, 'y_val.npy')
    Y_TEST_PATH = os.path.join(TFIDF_DIR, 'y_test.npy')
    tfidf_files_exist = all((os.path.exists(path) for path in [TFIDF_MODEL_PATH, X_TRAIN_TFIDF_PATH, X_VAL_TFIDF_PATH, X_TEST_TFIDF_PATH, Y_TRAIN_PATH, Y_VAL_PATH, Y_TEST_PATH]))
    if tfidf_files_exist:
        tfidf = joblib.load(TFIDF_MODEL_PATH)
        X_train_tfidf = __import__('scipy.sparse', fromlist=['load_npz']).load_npz(X_TRAIN_TFIDF_PATH)
        X_val_tfidf = __import__('scipy.sparse', fromlist=['load_npz']).load_npz(X_VAL_TFIDF_PATH)
        X_test_tfidf = __import__('scipy.sparse', fromlist=['load_npz']).load_npz(X_TEST_TFIDF_PATH)
        y_train = np.load(Y_TRAIN_PATH)
        y_val = np.load(Y_VAL_PATH)
        y_test = np.load(Y_TEST_PATH)
    else:
        tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
        X_train_tfidf = inference_fit_blocked()
        X_val_tfidf = tfidf.transform(val_df['cleaned_tweet'])
        X_test_tfidf = tfidf.transform(test_df['cleaned_tweet'])
        y_train = train_df['label'].to_numpy()
        y_val = val_df['label'].to_numpy()
        y_test = test_df['label'].to_numpy()
        joblib.dump(tfidf, TFIDF_MODEL_PATH)
        from scipy.sparse import save_npz
        save_npz(X_TRAIN_TFIDF_PATH, X_train_tfidf)
        save_npz(X_VAL_TFIDF_PATH, X_val_tfidf)
        save_npz(X_TEST_TFIDF_PATH, X_test_tfidf)
        np.save(Y_TRAIN_PATH, y_train)
        np.save(Y_VAL_PATH, y_val)
        np.save(Y_TEST_PATH, y_test)
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import LinearSVC
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.ensemble import RandomForestClassifier
    import joblib
    ML_MODEL_DIR = os.path.join(SAVE_DIR, '07_ml_models')
    os.makedirs(ML_MODEL_DIR, exist_ok=True)
    LR_PATH = os.path.join(ML_MODEL_DIR, 'logistic_regression.pkl')
    SVM_PATH = os.path.join(ML_MODEL_DIR, 'svm_calibrated.pkl')
    RF_PATH = os.path.join(ML_MODEL_DIR, 'random_forest.pkl')
    all_models_exist = all((os.path.exists(path) for path in [LR_PATH, SVM_PATH, RF_PATH]))
    if all_models_exist:
        lr = joblib.load(LR_PATH)
        svm_calibrated = joblib.load(SVM_PATH)
        rf = joblib.load(RF_PATH)
        ml_models = {'Logistic Regression': lr, 'SVM': svm_calibrated, 'Random Forest': rf}
    else:
        ml_models = {}
        lr = LogisticRegression(max_iter=1000, random_state=SEED)
        inference_fit_blocked()
        ml_models['Logistic Regression'] = lr
        joblib.dump(lr, LR_PATH)
        base_svm = LinearSVC(random_state=SEED, max_iter=3000)
        svm_calibrated = CalibratedClassifierCV(base_svm, cv=3)
        inference_fit_blocked()
        ml_models['SVM'] = svm_calibrated
        joblib.dump(svm_calibrated, SVM_PATH)
        rf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
        inference_fit_blocked()
        ml_models['Random Forest'] = rf
        joblib.dump(rf, RF_PATH)
    for name in ml_models:
        pass
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    import pickle
    MAX_WORDS = 10000
    MAX_LEN = 100
    EMBED_DIM = 64
    TOKENIZER_DIR = os.path.join(SAVE_DIR, '09_tokenizer')
    os.makedirs(TOKENIZER_DIR, exist_ok=True)
    TOKENIZER_PATH = os.path.join(TOKENIZER_DIR, 'keras_tokenizer.pkl')
    X_TRAIN_SEQ_PATH = os.path.join(TOKENIZER_DIR, 'X_train_seq.npy')
    X_VAL_SEQ_PATH = os.path.join(TOKENIZER_DIR, 'X_val_seq.npy')
    X_TEST_SEQ_PATH = os.path.join(TOKENIZER_DIR, 'X_test_seq.npy')
    PARAMS_PATH = os.path.join(TOKENIZER_DIR, 'tokenizer_params.pkl')
    tokenizer_files_exist = all((os.path.exists(path) for path in [TOKENIZER_PATH, X_TRAIN_SEQ_PATH, X_VAL_SEQ_PATH, X_TEST_SEQ_PATH, PARAMS_PATH]))
    if tokenizer_files_exist:
        with open(TOKENIZER_PATH, 'rb') as f:
            keras_tokenizer = pickle.load(f)
        X_train_seq = np.load(X_TRAIN_SEQ_PATH)
        X_val_seq = np.load(X_VAL_SEQ_PATH)
        X_test_seq = np.load(X_TEST_SEQ_PATH)
        with open(PARAMS_PATH, 'rb') as f:
            tokenizer_params = pickle.load(f)
        MAX_WORDS = tokenizer_params['MAX_WORDS']
        MAX_LEN = tokenizer_params['MAX_LEN']
        EMBED_DIM = tokenizer_params['EMBED_DIM']
    else:
        keras_tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
        inference_fit_blocked()

        def keras_encode(text_series):
            seq = keras_tokenizer.texts_to_sequences(text_series)
            return pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
        X_train_seq = keras_encode(train_df['cleaned_tweet'])
        X_val_seq = keras_encode(val_df['cleaned_tweet'])
        X_test_seq = keras_encode(test_df['cleaned_tweet'])
        with open(TOKENIZER_PATH, 'wb') as f:
            pickle.dump(keras_tokenizer, f)
        np.save(X_TRAIN_SEQ_PATH, X_train_seq)
        np.save(X_VAL_SEQ_PATH, X_val_seq)
        np.save(X_TEST_SEQ_PATH, X_test_seq)
        tokenizer_params = {'MAX_WORDS': MAX_WORDS, 'MAX_LEN': MAX_LEN, 'EMBED_DIM': EMBED_DIM}
        with open(PARAMS_PATH, 'wb') as f:
            pickle.dump(tokenizer_params, f)
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, Conv1D, GlobalMaxPooling1D
    from tensorflow.keras.callbacks import EarlyStopping

    def build_bilstm():
        model = Sequential([Embedding(input_dim=MAX_WORDS, output_dim=EMBED_DIM, input_length=MAX_LEN), Bidirectional(LSTM(64)), Dropout(0.3), Dense(32, activation='relu'), Dropout(0.2), Dense(1, activation='sigmoid')])
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        return model

    def build_textcnn():
        model = Sequential([Embedding(input_dim=MAX_WORDS, output_dim=EMBED_DIM, input_length=MAX_LEN), Conv1D(filters=128, kernel_size=3, activation='relu'), GlobalMaxPooling1D(), Dropout(0.3), Dense(64, activation='relu'), Dropout(0.2), Dense(1, activation='sigmoid')])
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        return model
    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    import os
    import pickle
    import numpy as np
    from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
    from tensorflow.keras.models import load_model
    BILSTM_DIR = os.path.join(SAVE_DIR, '10_dl_models', 'bilstm')
    os.makedirs(BILSTM_DIR, exist_ok=True)
    BILSTM_MODEL_PATH = os.path.join(BILSTM_DIR, 'best_model.keras')
    BILSTM_HISTORY_PATH = os.path.join(BILSTM_DIR, 'history.pkl')
    BILSTM_VAL_PROB_PATH = os.path.join(BILSTM_DIR, 'val_prob.npy')
    BILSTM_VAL_PRED_PATH = os.path.join(BILSTM_DIR, 'val_pred.npy')
    bilstm_checkpoint_exist = all((os.path.exists(path) for path in [BILSTM_MODEL_PATH, BILSTM_HISTORY_PATH, BILSTM_VAL_PROB_PATH, BILSTM_VAL_PRED_PATH]))
    if bilstm_checkpoint_exist:
        bilstm = load_model(BILSTM_MODEL_PATH)
        with open(BILSTM_HISTORY_PATH, 'rb') as f:
            history_bilstm_data = pickle.load(f)
        history_bilstm = type('History', (), {'history': history_bilstm_data})()
        bilstm_val_prob = np.load(BILSTM_VAL_PROB_PATH)
        bilstm_val_pred = np.load(BILSTM_VAL_PRED_PATH)
    else:
        bilstm = build_bilstm()
        bilstm_checkpoint = ModelCheckpoint(filepath=BILSTM_MODEL_PATH, monitor='val_loss', save_best_only=True, save_weights_only=False, verbose=1)
        early_stop_bilstm = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)
        history_bilstm = inference_fit_blocked()
        with open(BILSTM_HISTORY_PATH, 'wb') as f:
            pickle.dump(history_bilstm.history, f)
        bilstm = load_model(BILSTM_MODEL_PATH)
        bilstm_val_prob = bilstm.predict(X_val_seq, batch_size=256, verbose=1).ravel()
        bilstm_val_pred = (bilstm_val_prob >= 0.5).astype(int)
        np.save(BILSTM_VAL_PROB_PATH, bilstm_val_prob)
        np.save(BILSTM_VAL_PRED_PATH, bilstm_val_pred)
    bilstm_val_tweet, bilstm_val_user, _, _ = evaluate_tweet_and_user(val_df, bilstm_val_pred, bilstm_val_prob, 'Bi-LSTM', save=False)
    import os
    import pickle
    import numpy as np
    from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
    from tensorflow.keras.models import load_model
    TEXTCNN_DIR = os.path.join(SAVE_DIR, '10_dl_models', 'textcnn')
    os.makedirs(TEXTCNN_DIR, exist_ok=True)
    TEXTCNN_MODEL_PATH = os.path.join(TEXTCNN_DIR, 'best_model.keras')
    TEXTCNN_HISTORY_PATH = os.path.join(TEXTCNN_DIR, 'history.pkl')
    TEXTCNN_VAL_PROB_PATH = os.path.join(TEXTCNN_DIR, 'val_prob.npy')
    TEXTCNN_VAL_PRED_PATH = os.path.join(TEXTCNN_DIR, 'val_pred.npy')
    textcnn_checkpoint_exist = all((os.path.exists(path) for path in [TEXTCNN_MODEL_PATH, TEXTCNN_HISTORY_PATH, TEXTCNN_VAL_PROB_PATH, TEXTCNN_VAL_PRED_PATH]))
    if textcnn_checkpoint_exist:
        textcnn = load_model(TEXTCNN_MODEL_PATH)
        with open(TEXTCNN_HISTORY_PATH, 'rb') as f:
            history_textcnn_data = pickle.load(f)
        history_textcnn = type('History', (), {'history': history_textcnn_data})()
        textcnn_val_prob = np.load(TEXTCNN_VAL_PROB_PATH)
        textcnn_val_pred = np.load(TEXTCNN_VAL_PRED_PATH)
    else:
        textcnn = build_textcnn()
        textcnn_checkpoint = ModelCheckpoint(filepath=TEXTCNN_MODEL_PATH, monitor='val_loss', save_best_only=True, save_weights_only=False, verbose=1)
        early_stop_textcnn = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)
        history_textcnn = inference_fit_blocked()
        with open(TEXTCNN_HISTORY_PATH, 'wb') as f:
            pickle.dump(history_textcnn.history, f)
        textcnn = load_model(TEXTCNN_MODEL_PATH)
        textcnn_val_prob = textcnn.predict(X_val_seq, batch_size=256, verbose=1).ravel()
        textcnn_val_pred = (textcnn_val_prob >= 0.5).astype(int)
        np.save(TEXTCNN_VAL_PROB_PATH, textcnn_val_prob)
        np.save(TEXTCNN_VAL_PRED_PATH, textcnn_val_pred)
    textcnn_val_tweet, textcnn_val_user, _, _ = evaluate_tweet_and_user(val_df, textcnn_val_pred, textcnn_val_prob, 'TextCNN', save=False)
    from IPython.display import display
    all_results = []
    test_prediction_store = {}
    TEST_DIR = f'{SAVE_DIR}/14_stacking/test'
    os.makedirs(TEST_DIR, exist_ok=True)

    def add(name, pred, prob):
        assert len(pred) == len(test_df)
        evaluate_tweet_and_user(test_df, pred, prob, name, save=True)
        test_prediction_store[name] = {'pred': pred, 'prob': prob}
    for name, model in ml_models.items():
        add(name, model.predict(X_test_tfidf), model.predict_proba(X_test_tfidf)[:, 1])
    for name, m in [('Bi-LSTM', bilstm), ('TextCNN', textcnn)]:
        prob = m.predict(X_test_seq, batch_size=256, verbose=0).ravel()
        add(name, (prob >= 0.5).astype(int), prob)
    for n in ['DistilBERT', 'RoBERTa', 'BERTweet']:
        add(n, np.load(f'{TEST_DIR}/{n}_test_pred.npy'), np.load(f'{TEST_DIR}/{n}_test_prob.npy'))
    results_df = pd.DataFrame(all_results)
    RESULT_DIR = f'{SAVE_DIR}/15_final_results'
    os.makedirs(RESULT_DIR, exist_ok=True)
    results_df.to_csv(f'{RESULT_DIR}/all_results.csv', index=False)
    with open(f'{RESULT_DIR}/test_prediction_store.pkl', 'wb') as f:
        pickle.dump(test_prediction_store, f)
    cat = {'Logistic Regression': 'ML', 'SVM': 'ML', 'Random Forest': 'ML', 'Bi-LSTM': 'DL', 'TextCNN': 'DL', 'DistilBERT': 'Transformer', 'RoBERTa': 'Transformer', 'BERTweet': 'Transformer', 'Stacking Ensemble': 'Ensemble'}
    for lvl in ['tweet', 'user']:
        t = results_df[results_df['level'] == lvl].copy()
        t.insert(0, 'category', t['model'].map(cat))
        display(t.drop(columns='level').sort_values('macro_f1', ascending=False).round(4))
    archive = BASE / 'HOLDOUT_FINAL/15_final_results'
    archive.mkdir(parents=True, exist_ok=True)
    import shutil
    for filename in ['all_results.csv', 'test_prediction_store.pkl']:
        shutil.copyfile(Path(RESULT_DIR) / filename, archive / filename)
    return results_df


In [28]:
stage_functions = {
    'holdout_evaluation': evaluate_holdout_models
}
for selected, enabled in [(PREPARATION_STAGE,ENABLE_PREPROCESSING),(ANALYSIS_STAGE,ENABLE_ANALYSIS),(TRAINING_STAGE,ENABLE_TRAINING)]:
    if enabled is True and selected in stage_functions:
        stage_result = stage_functions[selected]()
        if stage_result is not None:
            display(stage_result)


## 8. Bootstrap for the Holdout Models

Stratified user bootstrap with 5,000 resamples.


In [29]:
def bootstrap_holdout_models():
    require_source_stage('holdout_ci', 'analysis', outputs=['17_conference_revision/user_level_bootstrap_CI.csv'], inputs=['15_final_results/test_prediction_store.pkl', '03_split/test_users.csv', '02_preprocessing/cleaned_data.pkl'])
    from pathlib import Path
    import pickle
    import numpy as np
    import pandas as pd
    from sklearn.metrics import roc_auc_score
    from tqdm.auto import tqdm
    BASE = Path('/content/drive_research/MyDrive/thesis/saved_results')
    OUT = BASE / '17_conference_revision'
    OUT.mkdir(parents=True, exist_ok=True)
    data = pd.read_pickle(BASE / '02_preprocessing/cleaned_data.pkl')[['username', 'label']]
    test_users = pd.read_csv(BASE / '03_split/test_users.csv')['username']
    test_data = data[data['username'].isin(test_users)].reset_index(drop=True)
    with open(BASE / '15_final_results/test_prediction_store.pkl', 'rb') as f:
        predictions = pickle.load(f)
    assert test_data.groupby('username')['label'].nunique().eq(1).all()

    def metrics(y, pred, prob):
        tn = np.sum((y == 0) & (pred == 0))
        fp = np.sum((y == 0) & (pred == 1))
        fn = np.sum((y == 1) & (pred == 0))
        tp = np.sum((y == 1) & (pred == 1))

        def div(a, b):
            return a / b if b else 0.0
        f1_asd = div(2 * tp, 2 * tp + fp + fn)
        f1_non = div(2 * tn, 2 * tn + fp + fn)
        return {'accuracy': div(tp + tn, len(y)), 'precision_ASD': div(tp, tp + fp), 'recall_ASD': div(tp, tp + fn), 'f1_ASD': f1_asd, 'macro_f1': (f1_asd + f1_non) / 2, 'recall_non_ASD': div(tn, tn + fp), 'f1_non_ASD': f1_non, 'roc_auc': roc_auc_score(y, prob)}
    rows = []
    B = 5000
    for name, arrays in predictions.items():
        pred = np.asarray(arrays['pred']).reshape(-1)
        prob = np.asarray(arrays['prob']).reshape(-1)
        assert len(pred) == len(prob) == len(test_data), 'Artifact validation failed.'
        assert np.isin(pred, [0, 1]).all(), 'Artifact validation failed.'
        assert np.isfinite(prob).all(), 'Artifact validation failed.'
        assert ((prob >= 0) & (prob <= 1)).all(), 'Artifact validation failed.'
        temp = test_data.copy()
        temp['pred'] = pred
        temp['prob'] = prob
        user = temp.groupby('username', sort=True).agg(label=('label', 'first'), vote=('pred', 'mean'), prob=('prob', 'mean'))
        y = user['label'].to_numpy(dtype=int)
        p = (user['vote'].to_numpy() >= 0.5).astype(int)
        s = user['prob'].to_numpy()
        assert len(user) == 61, 'Artifact validation failed.'
        assert (y == 0).sum() == 7, 'Artifact validation failed.'
        assert (y == 1).sum() == 54, 'Artifact validation failed.'
        point = metrics(y, p, s)
        groups = [np.flatnonzero(y == c) for c in [0, 1]]
        rng = np.random.default_rng(42)
        samples = []
        for _ in tqdm(range(B), desc=name):
            idx = np.concatenate([rng.choice(g, size=len(g), replace=True) for g in groups])
            samples.append(metrics(y[idx], p[idx], s[idx]))
        boot = pd.DataFrame(samples)
        for metric, value in point.items():
            rows.append({'model': name, 'metric': metric, 'estimate': value, 'CI_lower': boot[metric].quantile(0.025), 'CI_upper': boot[metric].quantile(0.975), 'n_users': len(user), 'n_non_ASD': int((y == 0).sum()), 'bootstrap_repetitions': B})
    results_ci = pd.DataFrame(rows)
    save_path = OUT / 'user_level_bootstrap_CI.csv'
    results_ci.to_csv(save_path, index=False)
    summary = results_ci[results_ci['metric'].isin(['macro_f1', 'roc_auc'])].copy()
    summary['score_and_CI'] = summary.apply(lambda r: f"{r['estimate']:.2f} [{r['CI_lower']:.2f}, {r['CI_upper']:.2f}]", axis=1)
    display(summary.pivot(index='model', columns='metric', values='score_and_CI'))
    return results_ci


In [30]:
stage_functions = {
    'holdout_ci': bootstrap_holdout_models
}
for selected, enabled in [(PREPARATION_STAGE,ENABLE_PREPROCESSING),(ANALYSIS_STAGE,ENABLE_ANALYSIS),(TRAINING_STAGE,ENABLE_TRAINING)]:
    if enabled is True and selected in stage_functions:
        stage_result = stage_functions[selected]()
        if stage_result is not None:
            display(stage_result)


## 9. Linguistic Baselines

Nine features × mean/SD/IQR. LR scaling is fitted on training users only; the threshold is fixed at 0.50.


In [31]:
def train_linguistic_baselines():
    require_source_stage('linguistic_baseline', 'training', outputs=['17_conference_revision/interpretable_baseline/*'], inputs=['17_conference_revision/linguistic_robustness/tweet_linguistic_features_corrected.pkl', '17_conference_revision/multiseed/seed_42/train_users.csv', '17_conference_revision/multiseed/seed_42/val_users.csv', '17_conference_revision/multiseed/seed_42/test_users.csv', '17_conference_revision/multiseed/seed_43/train_users.csv', '17_conference_revision/multiseed/seed_43/val_users.csv', '17_conference_revision/multiseed/seed_43/test_users.csv', '17_conference_revision/multiseed/seed_44/train_users.csv', '17_conference_revision/multiseed/seed_44/val_users.csv', '17_conference_revision/multiseed/seed_44/test_users.csv', '17_conference_revision/multiseed/seed_45/train_users.csv', '17_conference_revision/multiseed/seed_45/val_users.csv', '17_conference_revision/multiseed/seed_45/test_users.csv', '17_conference_revision/multiseed/seed_46/train_users.csv', '17_conference_revision/multiseed/seed_46/val_users.csv', '17_conference_revision/multiseed/seed_46/test_users.csv'])
    import json
    import platform
    from pathlib import Path
    import joblib
    import numpy as np
    import pandas as pd
    import sklearn
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
    BASE = Path('/content/drive_research/MyDrive/thesis/saved_results')
    SPLITS = BASE / '17_conference_revision/multiseed'
    OUT = BASE / '17_conference_revision/interpretable_baseline'
    OUT.mkdir(parents=True, exist_ok=True)
    FEATURES = ['tweet_length', 'vocab_richness', 'sentiment_polarity', 'sentiment_subjectivity', 'pronoun_i_rate', 'pronoun_we_rate', 'exclamation_rate', 'question_rate', 'readability']
    data = pd.read_pickle(BASE / '17_conference_revision/linguistic_robustness/tweet_linguistic_features_corrected.pkl')
    assert data[['username', 'label']].notna().all().all()
    assert data.groupby('username')['label'].nunique().eq(1).all()
    assert np.isfinite(data[FEATURES].to_numpy(dtype=float)).all()
    grouped = data.groupby('username', sort=True)
    means = grouped[FEATURES].mean().add_suffix('__mean')
    sds = grouped[FEATURES].std(ddof=1).add_suffix('__SD')
    iqrs = (grouped[FEATURES].quantile(0.75) - grouped[FEATURES].quantile(0.25)).add_suffix('__IQR')
    X_users = pd.concat([means, sds, iqrs], axis=1)
    y_users = grouped['label'].first().astype(int)
    assert X_users.shape == (244, 27)
    assert np.isfinite(X_users.to_numpy()).all()

    def load_users(path):
        table = pd.read_csv(path)
        if 'username' not in table.columns:
            raise ValueError('Required data or compatible artifacts are unavailable.')
        assert table['username'].notna().all()
        assert table['username'].is_unique
        return table['username'].tolist()

    def calculate_metrics(y_true, pred, prob):
        return {'accuracy': accuracy_score(y_true, pred), 'precision_ASD': precision_score(y_true, pred, pos_label=1, zero_division=0), 'recall_ASD': recall_score(y_true, pred, pos_label=1, zero_division=0), 'f1_ASD': f1_score(y_true, pred, pos_label=1, zero_division=0), 'macro_f1': f1_score(y_true, pred, average='macro', labels=[0, 1], zero_division=0), 'recall_non_ASD': recall_score(y_true, pred, pos_label=0, zero_division=0), 'f1_non_ASD': f1_score(y_true, pred, pos_label=0, zero_division=0), 'roc_auc': roc_auc_score(y_true, prob)}
    all_results = []
    coefficient_tables = []
    for seed in range(42, 47):
        seed_dir = SPLITS / f'seed_{seed}'
        train_users = load_users(seed_dir / 'train_users.csv')
        val_users = load_users(seed_dir / 'val_users.csv')
        test_users = load_users(seed_dir / 'test_users.csv')
        train_set, val_set, test_set = map(set, [train_users, val_users, test_users])
        assert not train_set & val_set
        assert not train_set & test_set
        assert not val_set & test_set
        assert train_set | val_set | test_set == set(X_users.index)
        assert (len(train_users), len(val_users), len(test_users)) == (146, 37, 61)
        X_train = X_users.loc[train_users]
        y_train = y_users.loc[train_users]
        X_test = X_users.loc[test_users]
        y_test = y_users.loc[test_users]
        assert y_train.nunique() == y_test.nunique() == 2
        models = {'LR (27 linguistic features)': make_pipeline(StandardScaler(), LogisticRegression(C=1.0, solver='lbfgs', max_iter=2000, class_weight='balanced', random_state=seed)), 'RF (27 linguistic features)': RandomForestClassifier(n_estimators=100, max_depth=None, min_samples_split=2, min_samples_leaf=1, max_features='sqrt', class_weight='balanced', random_state=seed, n_jobs=-1)}
        for model_name, model in models.items():
            model.fit(X_train, y_train)
            class_index = list(model.classes_).index(1)
            probabilities = model.predict_proba(X_test)[:, class_index]
            predictions = (probabilities >= 0.5).astype(int)
            metrics = calculate_metrics(y_test.to_numpy(), predictions, probabilities)
            all_results.append({'model': model_name, 'seed': seed, **metrics})
            tag = 'LR' if model_name.startswith('LR') else 'RF'
            pd.DataFrame({'username': test_users, 'label': y_test.to_numpy(), 'prediction': predictions, 'ASD_probability': probabilities}).to_csv(OUT / f'PRIVATE_{tag}_seed_{seed}_user_predictions.csv', index=False)
            joblib.dump(model, OUT / f'{tag}_seed_{seed}_model.joblib')
            if tag == 'LR':
                lr = model.named_steps['logisticregression']
                coefficient_tables.append(pd.DataFrame({'seed': seed, 'feature': X_users.columns, 'standardized_coefficient': lr.coef_[0], 'intercept': lr.intercept_[0]}))
            with open(OUT / f'{tag}_seed_{seed}_settings.json', 'w') as f:
                json.dump(model.get_params(deep=True), f, indent=2, default=str)
            pd.DataFrame(all_results).to_csv(OUT / 'linguistic_baseline_multiseed_results.csv', index=False)
    results = pd.DataFrame(all_results)
    metric_columns = ['accuracy', 'precision_ASD', 'recall_ASD', 'f1_ASD', 'macro_f1', 'recall_non_ASD', 'f1_non_ASD', 'roc_auc']
    summary = results.groupby('model')[metric_columns].agg(['mean', 'std'])
    summary.columns = [f'{metric}_{stat}' for metric, stat in summary.columns]
    summary = summary.reset_index()
    summary.to_csv(OUT / 'linguistic_baseline_multiseed_summary.csv', index=False)
    pd.concat(coefficient_tables, ignore_index=True).to_csv(OUT / 'LR_standardized_coefficients.csv', index=False)
    with open(OUT / 'experiment_settings.json', 'w') as f:
        json.dump({'python_version': platform.python_version(), 'sklearn_version': sklearn.__version__, 'pandas_version': pd.__version__, 'numpy_version': np.__version__, 'features': X_users.columns.tolist(), 'seeds': list(range(42, 47)), 'threshold': 0.5, 'training_users_per_seed': 146, 'validation_users_per_seed': 37, 'test_users_per_seed': 61, 'validation_used_for_tuning': False, 'imbalance_handling': 'training-only balanced class weights', 'SD_ddof': 1}, f, indent=2)
    return results


In [32]:
def bootstrap_linguistic_baselines():
    require_source_stage('linguistic_baseline_ci', 'analysis', outputs=['17_conference_revision/interpretable_baseline/linguistic_baseline_seed42_bootstrap_CI.csv'], inputs=['17_conference_revision/interpretable_baseline/PRIVATE_LR_seed_42_user_predictions.csv', '17_conference_revision/interpretable_baseline/PRIVATE_RF_seed_42_user_predictions.csv'])
    from pathlib import Path
    import numpy as np
    import pandas as pd
    from sklearn.metrics import roc_auc_score
    from tqdm.auto import tqdm
    OUT = Path('/content/drive_research/MyDrive/thesis/saved_results/17_conference_revision/interpretable_baseline')
    N_BOOTSTRAP = 5000
    BOOTSTRAP_SEED = 42

    def calculate_metrics(y, pred, prob):
        tp = np.sum((y == 1) & (pred == 1))
        tn = np.sum((y == 0) & (pred == 0))
        fp = np.sum((y == 0) & (pred == 1))
        fn = np.sum((y == 1) & (pred == 0))

        def divide(a, b):
            return float(a / b) if b else 0.0
        f1_asd = divide(2 * tp, 2 * tp + fp + fn)
        f1_non = divide(2 * tn, 2 * tn + fp + fn)
        return {'accuracy': divide(tp + tn, len(y)), 'precision_ASD': divide(tp, tp + fp), 'recall_ASD': divide(tp, tp + fn), 'f1_ASD': f1_asd, 'macro_f1': (f1_asd + f1_non) / 2, 'recall_non_ASD': divide(tn, tn + fp), 'f1_non_ASD': f1_non, 'roc_auc': roc_auc_score(y, prob)}
    all_intervals = []
    for tag in ['LR', 'RF']:
        path = OUT / f'PRIVATE_{tag}_seed_42_user_predictions.csv'
        predictions = pd.read_csv(path)
        required = ['username', 'label', 'prediction', 'ASD_probability']
        assert set(required).issubset(predictions.columns)
        assert predictions[required].notna().all().all()
        assert predictions['username'].is_unique
        assert len(predictions) == 61
        y = predictions['label'].to_numpy(dtype=int)
        pred = predictions['prediction'].to_numpy(dtype=int)
        prob = predictions['ASD_probability'].to_numpy(dtype=float)
        assert set(np.unique(y)) == {0, 1}
        assert set(np.unique(pred)).issubset({0, 1})
        assert np.isfinite(prob).all()
        assert ((prob >= 0) & (prob <= 1)).all()
        asd_indices = np.flatnonzero(y == 1)
        non_indices = np.flatnonzero(y == 0)
        assert len(asd_indices) == 54 and len(non_indices) == 7
        point_estimates = calculate_metrics(y, pred, prob)
        rng = np.random.default_rng(BOOTSTRAP_SEED)
        bootstrap_metrics = []
        for _ in tqdm(range(N_BOOTSTRAP), desc='Analysis'):
            sampled_indices = np.concatenate([rng.choice(asd_indices, size=len(asd_indices), replace=True), rng.choice(non_indices, size=len(non_indices), replace=True)])
            bootstrap_metrics.append(calculate_metrics(y[sampled_indices], pred[sampled_indices], prob[sampled_indices]))
        bootstrap_table = pd.DataFrame(bootstrap_metrics)
        for metric, estimate in point_estimates.items():
            low, high = np.percentile(bootstrap_table[metric], [2.5, 97.5])
            all_intervals.append({'model': f'{tag} (27 linguistic features)', 'seed': 42, 'metric': metric, 'estimate': estimate, 'CI_low': low, 'CI_high': high, 'n_bootstrap': N_BOOTSTRAP, 'bootstrap_seed': BOOTSTRAP_SEED, 'n_ASD': len(asd_indices), 'n_nonASD': len(non_indices)})
    intervals = pd.DataFrame(all_intervals)
    assert len(intervals) == 16
    assert np.isfinite(intervals[['estimate', 'CI_low', 'CI_high']].to_numpy()).all()
    intervals.to_csv(OUT / 'linguistic_baseline_seed42_bootstrap_CI.csv', index=False)
    for model, group in intervals.groupby('model', sort=False):
        for metric in ['macro_f1', 'roc_auc']:
            row = group.loc[group['metric'] == metric].iloc[0]


In [33]:
stage_functions = {
    'linguistic_baseline': train_linguistic_baselines,
    'linguistic_baseline_ci': bootstrap_linguistic_baselines
}
for selected, enabled in [(PREPARATION_STAGE,ENABLE_PREPROCESSING),(ANALYSIS_STAGE,ENABLE_ANALYSIS),(TRAINING_STAGE,ENABLE_TRAINING)]:
    if enabled is True and selected in stage_functions:
        stage_result = stage_functions[selected]()
        if stage_result is not None:
            display(stage_result)


### Baseline Results - Table IV

LR and RF use 27 user-level linguistic features. Show per-seed metrics, mean ± sample SD, and seed 42 intervals.


In [34]:
def linguistic_rf_predictions():
    table = read_report_csv(REVISION / 'interpretable_baseline/PRIVATE_RF_seed_42_user_predictions.csv')
    return pd.DataFrame([{'users': len(table),
                          'predicted_ASD': int(table['prediction'].eq(1).sum()),
                          'predicted_non_ASD': int(table['prediction'].eq(0).sum())}])


In [35]:
show_result('27-Feature Baselines',linguistic_baseline_metrics)
show_result('27-Feature Seed 42 Bootstrap Intervals',lambda:read_report_csv(REVISION/'interpretable_baseline/linguistic_baseline_seed42_bootstrap_CI.csv').round(4))
show_result('27-Feature RF Seed 42 Prediction Counts',linguistic_rf_predictions)


27-Feature Baselines


,model,seed,accuracy,precision_ASD,recall_ASD,f1_ASD,macro_f1,recall_non_ASD,f1_non_ASD,roc_auc
0,LR (27 linguistic features),42,0.6885,0.9730,0.6667,0.7912,0.5892,0.8571,0.3871,0.8254
1,RF (27 linguistic features),42,0.8852,0.8852,1.0000,0.9391,0.4696,0.0000,0.0000,0.8399
2,LR (27 linguistic features),43,0.7705,0.9762,0.7593,0.8542,0.6579,0.8571,0.4615,0.8254
3,RF (27 linguistic features),43,0.8852,0.8852,1.0000,0.9391,0.4696,0.0000,0.0000,0.7778
4,LR (27 linguistic features),44,0.8197,0.9216,0.8704,0.8952,0.6241,0.4286,0.3529,0.5529
5,RF (27 linguistic features),44,0.8852,0.8852,1.0000,0.9391,0.4696,0.0000,0.0000,0.6614
6,LR (27 linguistic features),45,0.8525,0.9245,0.9074,0.9159,0.6579,0.4286,0.4000,0.6799
7,RF (27 linguistic features),45,0.8852,0.8852,1.0000,0.9391,0.4696,0.0000,0.0000,0.7183
8,LR (27 linguistic features),46,0.8033,0.9375,0.8333,0.8824,0.6412,0.5714,0.4000,0.8730
9,RF (27 linguistic features),46,0.8852,0.8852,1.0000,0.9391,0.4696,0.0000,0.0000,0.8241


accuracy         precision_ASD         recall_ASD  \
                                mean     std          mean     std       mean   
model                                                                           
LR (27 linguistic features)   0.7869  0.0624        0.9466  0.0263     0.8074   
RF (27 linguistic features)   0.8852  0.0000        0.8852  0.0000     1.0000   

                                     f1_ASD         macro_f1          \
                                std    mean     std     mean     std   
model                                                                  
LR (27 linguistic features)  0.0959  0.8678  0.0483   0.6340  0.0287   
RF (27 linguistic features)  0.0000  0.9391  0.0000   0.4696  0.0000   

                            recall_non_ASD         f1_non_ASD         roc_auc  \
                                      mean     std       mean     std    mean   
model                                                                           
LR (27 linguistic features)         0.6286  0.2167     0.4003  0.0393  0.7513   
RF (27 linguistic features)         0.0000  0.0000     0.0000  0.0000  0.7643   

                                     
                                std  
model                                
LR (27 linguistic features)  0.1325  
RF (27 linguistic features)  0.0745

27-Feature Seed 42 Bootstrap Intervals


,model,seed,metric,estimate,CI_low,CI_high,n_bootstrap,bootstrap_seed,n_ASD,n_nonASD
0,LR (27 linguistic features),42,accuracy,0.6885,0.5738,0.8033,5000,42,54,7
1,LR (27 linguistic features),42,precision_ASD,0.9730,0.9167,1.0000,5000,42,54,7
2,LR (27 linguistic features),42,recall_ASD,0.6667,0.5370,0.7963,5000,42,54,7
3,LR (27 linguistic features),42,f1_ASD,0.7912,0.6905,0.8750,5000,42,54,7
4,LR (27 linguistic features),42,macro_f1,0.5892,0.4808,0.7053,5000,42,54,7
5,LR (27 linguistic features),42,recall_non_ASD,0.8571,0.5714,1.0000,5000,42,54,7
6,LR (27 linguistic features),42,f1_non_ASD,0.3871,0.2500,0.5217,5000,42,54,7
7,LR (27 linguistic features),42,roc_auc,0.8254,0.6587,0.9497,5000,42,54,7
8,RF (27 linguistic features),42,accuracy,0.8852,0.8852,0.8852,5000,42,54,7
9,RF (27 linguistic features),42,precision_ASD,0.8852,0.8852,0.8852,5000,42,54,7


27-Feature RF Seed 42 Prediction Counts


,users,predicted_ASD,predicted_non_ASD
0,61,61,0


,users,predicted_ASD,predicted_non_ASD
0,61,61,0
